In [ ]:

import sys
import platform
import torch

print("=== Python Environment ===")
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

print("\n=== PyTorch ===")
print("PyTorch version:", torch.__version__)

print("\n=== CUDA ===")
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print(
        "GPU memory (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    )
else:
    print("No GPU detected. Do not start model training yet.")


=== Python Environment ===
Python version: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39

=== PyTorch ===
PyTorch version: 2.11.0+cu130

=== CUDA ===
PyTorch CUDA build: 13.0
CUDA available: True
GPU name: Tesla T4
GPU count: 1
GPU memory (GB): 14.56


In [ ]:

import sys
import torch

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / (1024**3), 2
    ))


Python version: 3.13.15
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


In [ ]:

%pip install -q transformers peft accelerate bitsandbytes qwen-vl-utils


In [ ]:

# 1. Verify the environment
import torch
import transformers
import peft
import accelerate
import bitsandbytes
import qwen_vl_utils
from pathlib import Path

print("=== Library Check ===")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n=== Dataset File Check ===")

expected_files = [
    "images-20261010T124807Z-1-001.zip",
    "test-20261010T125233Z-1-001.zip",
    "labels (1).jsonl",
    "sequences.json",
    "score.py",
]

for filename in expected_files:
    matches = list(Path("/content").rglob(filename))
    if matches:
        print(f"FOUND {filename}: {matches[0]}")
    else:
        print(f"NOT FOUND: {filename}")


=== Library Check ===
Transformers: 5.18.0
PEFT: 0.21.1
Accelerate: 1.15.0
GPU available: True
GPU: Tesla T4

=== Dataset File Check ===
FOUND images-20261010T124807Z-1-001.zip: /content/images-20261010T124807Z-1-001.zip
FOUND test-20261010T125233Z-1-001.zip: /content/test-20261010T125233Z-1-001.zip
FOUND labels (1).jsonl: /content/labels (1).jsonl
FOUND sequences.json: /content/sequences.json
FOUND score.py: /content/score.py


In [ ]:

from pathlib import Path
from collections import Counter
import zipfile
import json
import shutil

ROOT = Path("/content")
DATA = ROOT / "manga_dataset"
DATA.mkdir(exist_ok=True)

# 1. Extract the archives
archives = [
    ROOT / "images-20261010T124807Z-1-001.zip",
    ROOT / "test-20261010T125233Z-1-001.zip",
]

for archive in archives:
    destination = DATA / archive.stem
    if not destination.exists():
        destination.mkdir(parents=True)
        with zipfile.ZipFile(archive) as z:
            z.extractall(destination)
        print("Extracted:", archive.name)
    else:
        print("Already extracted:", archive.name)

# 2. Inspect the labels
labels_path = ROOT / "labels (1).jsonl"
with labels_path.open(encoding="utf-8") as f:
    labels = [json.loads(line) for line in f if line.strip()]

print("\n=== Labels ===")
print("Labelled sequences:", len(labels))
print("First sequence ID:", labels[0]["sequence_id"])
print("Pages in first sequence:", len(labels[0]["pages"]))

# 3. Inspect metadata
with (ROOT / "sequences.json").open(encoding="utf-8") as f:
    metadata = json.load(f)

print("\n=== Metadata ===")
print("Total sequences:", len(metadata))
print("Split counts:", dict(Counter(
    row.get("split", "unspecified") for row in metadata
)))

# 4. Count extracted image files
images = [
    p for p in DATA.rglob("*")
    if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
]

print("\n=== Extracted images ===")
print("Image files found:", len(images))
print("Sample image paths:")
for p in images[:5]:
    print(p)


Extracted: images-20261010T124807Z-1-001.zip
Extracted: test-20261010T125233Z-1-001.zip

=== Labels ===
Labelled sequences: 80
First sequence ID: seq_2032620aa4e4ac7f
Pages in first sequence: 3

=== Metadata ===
Total sequences: 95
Split counts: {'development': 80, 'test': 15}

=== Extracted images ===
Image files found: 285
Sample image paths:
/content/manga_dataset/images-20261010T124807Z-1-001/images/seq_9032e5551cf1a9d2/03.png
/content/manga_dataset/images-20261010T124807Z-1-001/images/seq_9032e5551cf1a9d2/01.png
/content/manga_dataset/images-20261010T124807Z-1-001/images/seq_9032e5551cf1a9d2/02.png
/content/manga_dataset/images-20261010T124807Z-1-001/images/seq_4e5d126a68879ddf/03.png
/content/manga_dataset/images-20261010T124807Z-1-001/images/seq_4e5d126a68879ddf/01.png


In [ ]:

from pathlib import Path
import json

ROOT = Path("/content")
DATA = ROOT / "manga_dataset"

with (ROOT / "sequences.json").open(encoding="utf-8") as f:
    metadata = json.load(f)

# Build a lookup using each image's filename and sequence folder.
image_lookup = {}
for path in DATA.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}:
        image_lookup[(path.parent.name, path.name)] = path

missing = []
resolved = 0

for sequence in metadata:
    seq_id = sequence["sequence_id"]

    for image_path in sequence["images"]:
        filename = Path(image_path).name
        key = (seq_id, filename)

        if key in image_lookup:
            resolved += 1
        else:
            missing.append((seq_id, image_path))

print("=== Image Path Validation ===")
print("Sequences checked:", len(metadata))
print("Expected image paths:", sum(len(s["images"]) for s in metadata))
print("Successfully resolved:", resolved)
print("Missing paths:", len(missing))

if missing:
    print("\nFirst missing paths:")
    for item in missing[:10]:
        print(item)
else:
    print("SUCCESS: All metadata image paths resolve to extracted images.")


=== Image Path Validation ===
Sequences checked: 95
Expected image paths: 285
Successfully resolved: 285
Missing paths: 0
SUCCESS: All metadata image paths resolve to extracted images.


In [ ]:

import json
import random
from pathlib import Path

ROOT = Path("/content")
DATA = ROOT / "manga_dataset"

with (ROOT / "sequences.json").open(encoding="utf-8") as f:
    metadata = json.load(f)

# Keep only labelled development sequences.
dev_ids = [
    row["sequence_id"]
    for row in metadata
    if row["split"] == "development"
]

assert len(dev_ids) == 80, f"Expected 80 development sequences, got {len(dev_ids)}"

# Reproducible 64/16 split.
rng = random.Random(42)
shuffled_ids = dev_ids.copy()
rng.shuffle(shuffled_ids)

train_ids = sorted(shuffled_ids[:64])
val_ids = sorted(shuffled_ids[64:])

# Verify no overlap and correct counts.
assert len(train_ids) == 64
assert len(val_ids) == 16
assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids) | set(val_ids) == set(dev_ids)

# Save the split so future runs use exactly the same sequences.
split = {
    "seed": 42,
    "train_ids": train_ids,
    "validation_ids": val_ids,
}

split_path = DATA / "train_val_split_seed42.json"
split_path.write_text(
    json.dumps(split, indent=2) + "\n",
    encoding="utf-8"
)

print("Training sequences:", len(train_ids))
print("Validation sequences:", len(val_ids))
print("Overlap:", len(set(train_ids) & set(val_ids)))
print("Saved split:", split_path)
print("\nFirst 5 training IDs:", train_ids[:5])
print("First 5 validation IDs:", val_ids[:5])


Training sequences: 64
Validation sequences: 16
Overlap: 0
Saved split: /content/manga_dataset/train_val_split_seed42.json

First 5 training IDs: ['seq_06ec0acde37dffa4', 'seq_07838fb26cd1be54', 'seq_098e260f231aa789', 'seq_0dc2e3d874c804fc', 'seq_0ef2ec294597f572']
First 5 validation IDs: ['seq_032cbb4dc8686963', 'seq_2022b8c0e1f2d3c0', 'seq_2e4b837ead6b88ae', 'seq_2fae297a18af2a57', 'seq_44b9a316b2919cfe']


In [ ]:

import torch
import transformers
import bitsandbytes as bnb
from transformers import AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print("=== Model Setup Check ===")
print("Model:", MODEL_ID)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# Check whether Transformers can recognize this model's configuration.
from transformers import AutoConfig

try:
    config = AutoConfig.from_pretrained(MODEL_ID)
    print("Model configuration loaded:", config.model_type)
except Exception as e:
    print("MODEL CONFIGURATION ERROR:")
    print(type(e).__name__, str(e)[:1000])

# Check that 4-bit quantization can be configured.
try:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    print("4-bit configuration created successfully.")
except Exception as e:
    print("4-BIT CONFIGURATION ERROR:")
    print(type(e).__name__, str(e)[:1000])


=== Model Setup Check ===
Model: Qwen/Qwen2.5-VL-3B-Instruct
Transformers: 5.18.0
PyTorch: 2.11.0+cu130
GPU: Tesla T4
Model configuration loaded: qwen2_5_vl
4-bit configuration created successfully.


In [ ]:

import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model in 4-bit mode...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("\nModel loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory allocated (GB):", round(
    torch.cuda.memory_allocated(0) / (1024**3), 2
))


Loading model in 4-bit mode...


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]


Model loaded successfully!
GPU: Tesla T4
GPU memory allocated (GB): 2.25


In [ ]:

import json
from pathlib import Path
import torch
from qwen_vl_utils import process_vision_info

# Load dataset metadata and the saved validation split
with open("/content/sequences.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open("/content/manga_dataset/train_val_split_seed42.json", "r", encoding="utf-8") as f:
    split = json.load(f)

val_ids = split["validation_ids"]
print("Validation sequences:", len(val_ids))

# Find the images using their sequence ID and filename
image_root = Path("/content/manga_dataset")
image_lookup = {}

for path in image_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}:
        image_lookup[(path.parent.name, path.name)] = path

# Locate the first validation sequence
sequence_id = val_ids[0]
record = next(row for row in metadata if row["sequence_id"] == sequence_id)

image_paths = []
for image_entry in record["images"]:
    filename = Path(image_entry).name
    matches = [
        path for (folder, name), path in image_lookup.items()
        if name == filename and folder == sequence_id
    ]
    if not matches:
        # Fall back to matching the filename anywhere in the extracted dataset
        matches = [path for path in image_root.rglob(filename) if path.is_file()]
    if not matches:
        raise FileNotFoundError(f"Could not find image: {filename}")
    image_paths.append(matches[0])

assert len(image_paths) == 3, f"Expected 3 images, found {len(image_paths)}"
print("Testing sequence:", sequence_id)
print("Images:")
for path in image_paths:
    print(" ", path)

instruction = """
Transcribe the story text from these THREE consecutive English manga pages.

IMPORTANT RULES:
1. Keep every line on the page where it actually appears.
2. Preserve the intended reading order within each page.
3. Include dialogue, thoughts, narration, vocalizations, and punctuation-only speech balloons.
4. Exclude sound effects, credits, labels, signs, and other non-story text.
5. Use consistent speaker IDs across all three pages.
6. Use speaker IDs such as character1, character2, character3, etc.
7. Use "UNKNOWN" when the speaker cannot be identified confidently.
8. Do not invent dialogue or move text between pages.
9. If a page has no qualifying story text, return an empty list for that page.

Return ONLY valid JSON in this exact structure:
{
  "pages": [
    [{"speaker": "character1", "text": "Example dialogue."}],
    [],
    []
  ]
}

There must be exactly three page arrays, in input image order.
Each line item must contain only "speaker" and "text".
Do not include Markdown fences or explanations.
"""


messages = [
    {
        "role": "user",
        "content": (
            [{"type": "image", "image": str(path)} for path in image_paths]
            + [{"type": "text", "text": instruction}]
        ),
    }
]

prompt = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[prompt],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=False,
    )

new_tokens = output_ids[:, inputs.input_ids.shape[1]:]
generated_text = processor.batch_decode(
    new_tokens,
    skip_special_tokens=True,
)[0]

print("\n----- MODEL OUTPUT -----\n")
print(generated_text)


Validation sequences: 16
Testing sequence: seq_032cbb4dc8686963
Images:
  /content/manga_dataset/images-20261010T124807Z-1-001/images/seq_032cbb4dc8686963/01.png
  /content/manga_dataset/images-20261010T124807Z-1-001/images/seq_032cbb4dc8686963/02.png
  /content/manga_dataset/images-20261010T124807Z-1-001/images/seq_032cbb4dc8686963/03.png

----- MODEL OUTPUT -----

```json
{
  "pages": [
    {
      "speaker": "character1",
      "text": "YOU'RE THE ONLY ONE."
    },
    {
      "speaker": "character2",
      "text": "TH..."
    },
    {
      "speaker": "character1",
      "text": "THE ONLY ONE...?"
    },
    {
      "speaker": "character2",
      "text": "I... I CAN'T ASK ANYONE ELSE."
    },
    {
      "speaker": "character1",
      "text": "WOW..."
    },
    {
      "speaker": "character2",
      "text": "I... IT HAS TO BE YOU..."
    },
    {
      "speaker": "character1",
      "text": "YOU WILL!?"
    },
    {
      "speaker": "character2",
      "text": "JUST THIS TIME..."


In [ ]:
print("Raw output captured:", "generated_text" in globals())
print("Output length:", len(generated_text) if "generated_text" in globals() else 0)

Raw output captured: True
Output length: 1524


In [ ]:

import json
from pathlib import Path

labels_path = Path("/content/labels (1).jsonl")
submission_path = Path("/content/sample_submission.jsonl")

# 1. Inspect actual ground-truth annotations
print("=" * 60)
print("GROUND-TRUTH LABELS")
print("=" * 60)

with labels_path.open("r", encoding="utf-8") as f:
    first_label = json.loads(next(line for line in f if line.strip()))

print("Top-level keys:", list(first_label.keys()))
print("Sequence ID:", first_label.get("sequence_id"))
print("Number of pages:", len(first_label.get("pages", [])))
print("Lines per page:", [len(page) for page in first_label.get("pages", [])])

for page_number, page in enumerate(first_label.get("pages", []), start=1):
    print(f"\nPage {page_number}:")
    for line_number, item in enumerate(page, start=1):
        print(f"  Line {line_number}: {item}")

# 2. Inspect the expected submission format
print("\n" + "=" * 60)
print("SAMPLE SUBMISSION")
print("=" * 60)

if submission_path.exists():
    with submission_path.open("r", encoding="utf-8") as f:
        first_submission = json.loads(next(line for line in f if line.strip()))

    print("Top-level keys:", list(first_submission.keys()))
    print("Sequence ID:", first_submission.get("sequence_id"))
    print("Number of pages:", len(first_submission.get("pages", [])))
    print("Example structure:")
    print(json.dumps(first_submission, indent=2, ensure_ascii=False)[:2500])
else:
    print("sample_submission.jsonl was not found at:", submission_path)


GROUND-TRUTH LABELS
Top-level keys: ['sequence_id', 'pages']
Sequence ID: seq_2032620aa4e4ac7f
Number of pages: 3
Lines per page: [5, 0, 6]

Page 1:
  Line 1: {'speaker': 'character1', 'text': 'HUH? DOES KOMABA SEEM DIFFERENT THAN USUAL?'}
  Line 2: {'speaker': 'character2', 'text': "HE'S DOING REALLY WELL. IT LOOKS LIKE HE'S BEEN WORKING HARD FOR THE FALL TOURNAMENT."}
  Line 3: {'speaker': 'UNKNOWN', 'text': "THAT'S THE WAY!"}
  Line 4: {'speaker': 'character1', 'text': "NO, THAT'S NOT IT. IT'S LIKE HE'S..."}
  Line 5: {'speaker': 'character1', 'text': 'OUT FOR BLOOD.'}

Page 2:

Page 3:
  Line 1: {'speaker': 'UNKNOWN', 'text': 'WOAH!!'}
  Line 2: {'speaker': 'character3', 'text': 'OKAWA-SENPAI, YOU LOOK SO COOL!'}
  Line 3: {'speaker': 'character4', 'text': 'RIGHT HERE! THIS IS WHERE IT HAPPENS!'}
  Line 4: {'speaker': 'UNKNOWN', 'text': 'SO CLOSE!!'}
  Line 5: {'speaker': 'character5', 'text': "DO YOU HAVE TO KEEP WATCHING THAT? IT'S EMBARRASSING."}
  Line 6: {'speaker': 'character

In [ ]:

import json
import re
import subprocess
import sys
import time
from pathlib import Path

import torch
from qwen_vl_utils import process_vision_info

# --------------------------------------------------
# 1. Verify required files and load validation IDs
# --------------------------------------------------
BASE = Path("/content")
DATA_ROOT = BASE / "manga_dataset"
LABELS_PATH = BASE / "labels (1).jsonl"
METADATA_PATH = BASE / "sequences.json"
SPLIT_PATH = DATA_ROOT / "train_val_split_seed42.json"
SCORER_PATH = BASE / "score.py"

for path in [LABELS_PATH, METADATA_PATH, SPLIT_PATH, SCORER_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

if "model" not in globals() or "processor" not in globals():
    raise RuntimeError("Model or processor is missing. Do not reload yet; tell me this error.")

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

with SPLIT_PATH.open("r", encoding="utf-8") as f:
    split = json.load(f)

with LABELS_PATH.open("r", encoding="utf-8") as f:
    labels = [json.loads(line) for line in f if line.strip()]

validation_ids = split["validation_ids"]
label_by_id = {row["sequence_id"]: row for row in labels}
metadata_by_id = {row["sequence_id"]: row for row in metadata}

assert len(validation_ids) == 16, f"Expected 16 validation IDs, got {len(validation_ids)}"
assert all(sid in label_by_id for sid in validation_ids), "Some validation labels are missing."
assert all(sid in metadata_by_id for sid in validation_ids), "Some validation metadata is missing."

print("Validation sequences:", len(validation_ids))
print("Model:", "already loaded")
print("Starting baseline inference...")

# --------------------------------------------------
# 2. Resolve all image paths
# --------------------------------------------------
image_lookup = {}
for path in DATA_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}:
        image_lookup[(path.parent.name, path.name)] = path

def get_sequence_images(sequence_id):
    entries = metadata_by_id[sequence_id]["images"]
    paths = []

    for entry in entries:
        filename = Path(entry).name
        path = image_lookup.get((sequence_id, filename))
        if path is None:
            raise FileNotFoundError(
                f"Missing image for {sequence_id}: {filename}"
            )
        paths.append(path)

    if len(paths) != 3:
        raise ValueError(f"{sequence_id} has {len(paths)} images, not 3.")

    return paths

# --------------------------------------------------
# 3. Helpers for inference and JSON parsing
# --------------------------------------------------
def generate_text(image_paths, instruction, max_new_tokens=1200):
    content = [{"type": "image", "image": str(p)} for p in image_paths]
    content.append({"type": "text", "text": instruction})
    messages = [{"role": "user", "content": content}]

    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = output_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(
        new_tokens, skip_special_tokens=True
    )[0].strip()


def parse_json_response(raw):
    # Accept JSON even if the model wraps it in Markdown fences.
    text = re.sub(r"^\s*```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    start = text.find("{")
    end = text.rfind("}")

    if start < 0 or end < start:
        return None

    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def clean_lines(lines):
    if not isinstance(lines, list):
        return None

    cleaned = []
    for item in lines:
        if not isinstance(item, dict):
            return None
        if not isinstance(item.get("speaker"), str):
            return None
        if not isinstance(item.get("text"), str):
            return None

        cleaned.append({
            "speaker": item["speaker"],
            "text": item["text"],
        })

    return cleaned


def parse_three_pages(raw):
    obj = parse_json_response(raw)
    if not isinstance(obj, dict):
        return None

    pages = obj.get("pages")
    if not isinstance(pages, list) or len(pages) != 3:
        return None

    cleaned_pages = []
    for page in pages:
        cleaned = clean_lines(page)
        if cleaned is None:
            return None
        cleaned_pages.append(cleaned)

    return cleaned_pages


def parse_single_page(raw):
    obj = parse_json_response(raw)
    if not isinstance(obj, dict):
        return None

    # The single-page prompt specifically requests {"lines": [...]}.
    lines = obj.get("lines")
    return clean_lines(lines)


FULL_PROMPT = """
Transcribe story text from these three consecutive English manga pages.
The images are page 1, page 2, and page 3, in that order.

Include dialogue, thoughts, narration, vocalizations, and punctuation-only
speech balloons. Exclude sound effects, credits, labels, signs, and other
non-story text. Preserve reading order and page boundaries exactly.
Do not invent text or move text from one page to another.
Use speaker IDs consistently across pages, such as character1, character2,
and so on. Use UNKNOWN when the speaker cannot be identified confidently.
A page with no qualifying story text must be [].

Return ONLY valid JSON in this exact format:
{
  "pages": [
    [{"speaker": "character1", "text": "Example."}],
    [],
    []
  ]
}
There must be exactly three arrays inside "pages".
Every line item must contain only "speaker" and "text".
"""

SINGLE_PAGE_PROMPT = """
Transcribe only the story text visible on this ONE manga page.
Include dialogue, thoughts, narration, vocalizations, and punctuation-only
speech balloons. Exclude sound effects, credits, labels, signs, and other
non-story text. Preserve reading order. Do not invent text.
Use speaker IDs such as character1, character2, etc., and UNKNOWN when unsure.
Return ONLY JSON in this format:
{"lines": [{"speaker": "character1", "text": "Example."}]}
If there is no qualifying story text, return {"lines": []}.
"""

# --------------------------------------------------
# 4. Run inference on all 16 validation sequences
# --------------------------------------------------
predictions = []
raw_records = []
fallback_count = 0
failed_pages = 0
start_time = time.time()

for index, sequence_id in enumerate(validation_ids, start=1):
    image_paths = get_sequence_images(sequence_id)
    print(f"\n[{index}/{len(validation_ids)}] {sequence_id}")

    raw = generate_text(image_paths, FULL_PROMPT, max_new_tokens=1400)
    pages = parse_three_pages(raw)
    method = "three_page"

    # If the model doesn't produce three page arrays, retry page by page.
    if pages is None:
        fallback_count += 1
        method = "page_by_page_fallback"
        pages = []

        for page_index, image_path in enumerate(image_paths, start=1):
            try:
                page_raw = generate_text(
                    [image_path], SINGLE_PAGE_PROMPT, max_new_tokens=500
                )
                page_lines = parse_single_page(page_raw)

                if page_lines is None:
                    print(f"  Warning: page {page_index} output could not be parsed.")
                    page_lines = []
                    failed_pages += 1

                pages.append(page_lines)
            except Exception as exc:
                print(f"  Warning: page {page_index} failed: {exc}")
                pages.append([])
                failed_pages += 1

    assert len(pages) == 3

    predictions.append({
        "sequence_id": sequence_id,
        "pages": pages,
    })
    raw_records.append({
        "sequence_id": sequence_id,
        "method": method,
        "raw_output": raw,
    })

    print("  Method:", method)
    print("  Predicted lines per page:", [len(p) for p in pages])

    # Save progress after every sequence, in case the session disconnects.
    with (BASE / "baseline_predictions.jsonl").open("w", encoding="utf-8") as f:
        for row in predictions:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with (BASE / "baseline_raw_outputs.jsonl").open("w", encoding="utf-8") as f:
        for row in raw_records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

# --------------------------------------------------
# 5. Save validation references and score predictions
# --------------------------------------------------
references = [label_by_id[sid] for sid in validation_ids]

with (BASE / "validation_references.jsonl").open("w", encoding="utf-8") as f:
    for row in references:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\nInference complete.")
print("Sequences predicted:", len(predictions))
print("Sequences requiring page-by-page fallback:", fallback_count)
print("Pages that could not be parsed:", failed_pages)
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Predictions saved to:", BASE / "baseline_predictions.jsonl")
print("Raw outputs saved to:", BASE / "baseline_raw_outputs.jsonl")

# Use the official scorer; do not calculate substitute metrics.
score_path = BASE / "baseline_scores.json"
command = [
    sys.executable, str(SCORER_PATH),
    "--references", str(BASE / "validation_references.jsonl"),
    "--predictions", str(BASE / "baseline_predictions.jsonl"),
    "--output", str(score_path),
]

print("\nRunning official scorer...")
result = subprocess.run(command, capture_output=True, text=True)

print(result.stdout)
if result.stderr.strip():
    print("Scorer messages:")
    print(result.stderr[-5000:])

if result.returncode == 0:
    print("Official scorer completed.")
    if score_path.exists():
        print(score_path.read_text(encoding="utf-8"))
else:
    print("SCORER FAILED with exit code:", result.returncode)
    print("Keep the saved predictions. Send me the scorer error; do not rerun inference.")


Validation sequences: 16
Model: already loaded
Starting baseline inference...

[1/16] seq_032cbb4dc8686963
  Method: page_by_page_fallback
  Predicted lines per page: [9, 9, 7]

[2/16] seq_2022b8c0e1f2d3c0
  Method: three_page
  Predicted lines per page: [1, 2, 1]

[3/16] seq_2e4b837ead6b88ae
  Method: page_by_page_fallback
  Predicted lines per page: [8, 0, 6]

[4/16] seq_2fae297a18af2a57
  Method: page_by_page_fallback
  Predicted lines per page: [9, 7, 6]

[5/16] seq_44b9a316b2919cfe
  Method: page_by_page_fallback
  Predicted lines per page: [9, 5, 6]

[6/16] seq_535e0754d805105b
  Method: page_by_page_fallback
  Predicted lines per page: [0, 10, 8]

[7/16] seq_5e4a9d63617d62bf
  Method: page_by_page_fallback
  Predicted lines per page: [4, 7, 7]

[8/16] seq_5f9ee35846b7e635
  Method: page_by_page_fallback
  Predicted lines per page: [6, 5, 3]

[9/16] seq_888a2728b36f1745
  Method: page_by_page_fallback
  Predicted lines per page: [5, 2, 5]

[10/16] seq_9032e5551cf1a9d2
  Method: p

In [ ]:
import json
from pathlib import Path

split_path = Path("/content/manga_dataset/train_val_split_seed42.json")
labels_path = Path("/content/labels (1).jsonl")
predictions_path = Path("/content/baseline_predictions.jsonl")
scores_path = Path("/content/baseline_scores.json")

assert split_path.exists(), f"Missing: {split_path}"
assert labels_path.exists(), f"Missing: {labels_path}"
assert predictions_path.exists(), "Baseline predictions not found"
assert scores_path.exists(), "Baseline scores not found"

with open(split_path) as f:
    split = json.load(f)

print("Split keys:", list(split.keys()))

train_ids = split.get("train_ids", split.get("training_ids"))
validation_ids = split.get("validation_ids")

assert train_ids is not None, "Could not find training IDs in split file"
assert validation_ids is not None, "Could not find validation IDs in split file"
assert len(set(train_ids) & set(validation_ids)) == 0, "Train/validation overlap!"

print("Training sequences:", len(train_ids))
print("Validation sequences:", len(validation_ids))
print("Overlap:", len(set(train_ids) & set(validation_ids)))
print("Baseline predictions:", sum(
    1 for line in open(predictions_path) if line.strip()
))
print("Baseline score file exists:", scores_path.exists())
print("Model loaded in memory:", "model" in globals())
print("Processor loaded in memory:", "processor" in globals())

Split keys: ['seed', 'train_ids', 'validation_ids']
Training sequences: 64
Validation sequences: 16
Overlap: 0
Baseline predictions: 16
Baseline score file exists: True
Model loaded in memory: True
Processor loaded in memory: True


below code just checks the pages which failed parsing and gets their raw output

In [ ]:
import json

path = "/content/baseline_raw_outputs.jsonl"

with open(path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

for record in records:
    raw = json.dumps(record, ensure_ascii=False)
    if (
        "seq_2e4b837ead6b88ae" in raw
        or "seq_535e0754d805105b" in raw
    ):
        print("\n" + "=" * 70)
        print(raw[:12000])


{"sequence_id": "seq_2e4b837ead6b88ae", "method": "page_by_page_fallback", "raw_output": "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [ ]:
import torch

# Confirm the model and processor are still loaded
assert "model" in globals(), "Model is not loaded."
assert "processor" in globals(), "Processor is not loaded."
assert torch.cuda.is_available(), "GPU is not available."

# Inspect the model structure before changing training settings
print("Model class:", type(model).__name__)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# Find and freeze the vision encoder
vision_encoder = getattr(model.model, "visual", None)

assert vision_encoder is not None, (
    "Could not find model.model.visual; stop here and share the output."
)

for parameter in vision_encoder.parameters():
    parameter.requires_grad = False

# Limit each image to at most 512 × 512 pixels during training
image_processor = processor.image_processor
assert hasattr(image_processor, "max_pixels"), (
    "Image processor has no max_pixels setting; stop here."
)

original_max_pixels = image_processor.max_pixels
image_processor.max_pixels = 512 * 512

print("\nVision encoder frozen:", all(
    not p.requires_grad for p in vision_encoder.parameters()
))
print("Original image pixel cap:", original_max_pixels)
print("Training image pixel cap:", image_processor.max_pixels)
print("Model preparation check complete.")

Model class: Qwen2_5_VLForConditionalGeneration
GPU: Tesla T4
GPU memory allocated: 2.69 GB
GPU memory reserved:  12.00 GB


AssertionError: Image processor has no max_pixels setting; stop here.

In [ ]:
image_processor = processor.image_processor

print("Processor class:", type(image_processor).__name__)
print("Image size:", getattr(image_processor, "size", None))
print("Min pixels:", getattr(image_processor, "min_pixels", None))
print("Max pixels:", getattr(image_processor, "max_pixels", None))
print("Processor attributes:", [
    name for name in vars(image_processor)
    if "pixel" in name.lower() or "size" in name.lower()
])

Processor class: Qwen2VLImageProcessor
Image size: SizeDict(height=None, width=None, longest_edge=12845056, shortest_edge=3136, max_height=None, max_width=None, min_pixels=None, max_pixels=None)
Min pixels: None
Max pixels: None
Processor attributes: ['size', 'patch_size', 'temporal_patch_size', 'merge_size', 'do_resize', 'crop_size', 'pad_size']


In [ ]:
# Set a 512 × 512 pixel budget using the processor's actual configuration
processor.image_processor.size["max_pixels"] = 512 * 512

print("Updated image processor size:", processor.image_processor.size)
print("Vision encoder frozen:", all(
    not p.requires_grad
    for p in model.model.visual.parameters()
))
print("Ready for LoRA setup.")

Updated image processor size: SizeDict(height=None, width=None, longest_edge=12845056, shortest_edge=3136, max_height=None, max_width=None, min_pixels=None, max_pixels=262144)
Vision encoder frozen: True
Ready for LoRA setup.


In [ ]:
import torch
import bitsandbytes as bnb

# Check that the base model is still quantized
print("4-bit quantization:", getattr(model, "is_loaded_in_4bit", False))
print("Vision encoder frozen:", all(
    not p.requires_grad for p in model.model.visual.parameters()
))

# Inspect the actual projection layers available for LoRA
target_suffixes = (
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
)

module_counts = {name: 0 for name in target_suffixes}

for name, module in model.named_modules():
    for suffix in target_suffixes:
        if name.endswith(suffix):
            module_counts[suffix] += 1

print("\nLoRA target layer counts:")
for name, count in module_counts.items():
    print(f"{name}: {count}")

assert all(count > 0 for count in module_counts.values()), (
    "Some expected projection layers were not found. Stop and share the output."
)

print("\nLayer verification complete.")

4-bit quantization: True
Vision encoder frozen: True

LoRA target layer counts:
q_proj: 36
k_proj: 36
v_proj: 36
o_proj: 36
gate_proj: 68
up_proj: 68
down_proj: 68

Layer verification complete.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantized model for adapter training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# Configure trainable LoRA adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nQLoRA adapters attached.")

trainable params: 18,576,384 || all params: 3,773,199,360 || trainable%: 0.4923

QLoRA adapters attached.


In [ ]:
import json
from pathlib import Path

with open("/content/manga_dataset/train_val_split_seed42.json") as f:
    split = json.load(f)

with open("/content/labels (1).jsonl", encoding="utf-8") as f:
    labels = {
        record["sequence_id"]: record
        for record in (json.loads(line) for line in f if line.strip())
    }

train_ids = split["train_ids"]
validation_ids = split["validation_ids"]

print("Training IDs:", len(train_ids))
print("Validation IDs:", len(validation_ids))
print("Training IDs with labels:", sum(sid in labels for sid in train_ids))
print("Validation IDs with labels:", sum(sid in labels for sid in validation_ids))

example = labels[train_ids[0]]
print("\nExample sequence:", example["sequence_id"])
print("Pages:", len(example["pages"]))
print("Lines per page:", [len(page) for page in example["pages"]])
print("Example labelled line:", next(
    (line for page in example["pages"] for line in page),
    "No dialogue lines"
))

Training IDs: 64
Validation IDs: 16
Training IDs with labels: 64
Validation IDs with labels: 16

Example sequence: seq_06ec0acde37dffa4
Pages: 3
Lines per page: [7, 9, 7]
Example labelled line: {'speaker': 'character1', 'text': 'UM... OUR NEXT CLASS WILL BE COOKING PRACTICE.'}


we are now verifying image paths

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/manga_dataset")
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}

train_id_set = set(train_ids)
validation_id_set = set(validation_ids)
image_paths_by_sequence = {}

for path in DATA_ROOT.rglob("*"):
    if (
        path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
        and path.parent.name in train_id_set | validation_id_set
    ):
        image_paths_by_sequence.setdefault(path.parent.name, []).append(path)

# Sort each sequence's pages consistently
for sid in image_paths_by_sequence:
    image_paths_by_sequence[sid].sort(key=lambda p: p.name)

train_counts = {
    sid: len(image_paths_by_sequence.get(sid, []))
    for sid in train_ids
}
validation_counts = {
    sid: len(image_paths_by_sequence.get(sid, []))
    for sid in validation_ids
}

print("Training sequences with 3 images:",
      sum(count == 3 for count in train_counts.values()), "/", len(train_ids))
print("Validation sequences with 3 images:",
      sum(count == 3 for count in validation_counts.values()), "/", len(validation_ids))

bad_train = {sid: n for sid, n in train_counts.items() if n != 3}
bad_validation = {sid: n for sid, n in validation_counts.items() if n != 3}

print("Training sequences with incorrect image count:", bad_train)
print("Validation sequences with incorrect image count:", bad_validation)

if not bad_train and not bad_validation:
    example_id = train_ids[0]
    print("\nExample training sequence:", example_id)
    for p in image_paths_by_sequence[example_id]:
        print(p)
    print("\nImage-path verification passed.")
else:
    print("\nImage-path verification failed; do not start training yet.")

Training sequences with 3 images: 64 / 64
Validation sequences with 3 images: 16 / 16
Training sequences with incorrect image count: {}
Validation sequences with incorrect image count: {}

Example training sequence: seq_06ec0acde37dffa4
/content/manga_dataset/images-20261010T124807Z-1-001/images/seq_06ec0acde37dffa4/01.png
/content/manga_dataset/images-20261010T124807Z-1-001/images/seq_06ec0acde37dffa4/02.png
/content/manga_dataset/images-20261010T124807Z-1-001/images/seq_06ec0acde37dffa4/03.png

Image-path verification passed.


In [ ]:
import json
import gc
import torch
from qwen_vl_utils import process_vision_info

# Use one labelled TRAINING example only
sid = train_ids[0]
record = labels[sid]
paths = image_paths_by_sequence[sid]

instruction = """
Transcribe the English dialogue, thoughts, narration, and vocalizations
from these three consecutive manga pages, in reading order.
Exclude sound effects, credits, signs, and other non-dialogue text.
Return only valid JSON with exactly three page arrays:
{"pages": [[{"speaker":"character1","text":"..."}], [], []]}
Each line must contain speaker and text. Preserve punctuation and page
boundaries. Keep speaker labels consistent across all three pages.
Use UNKNOWN when the speaker cannot be identified. Do not add explanations.
"""

user_message = {
    "role": "user",
    "content": (
        [{"type": "image", "image": str(p)} for p in paths]
        + [{"type": "text", "text": instruction}]
    )
}

target = json.dumps(
    {"pages": record["pages"]},
    ensure_ascii=False
)

messages = [
    user_message,
    {"role": "assistant", "content": [{"type": "text", "text": target}]}
]

# Build the complete conversation and the prompt-only version
full_text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=False
)
prompt_text = processor.apply_chat_template(
    [user_message], tokenize=False, add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

full_inputs = processor(
    text=[full_text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

prompt_inputs = processor(
    text=[prompt_text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

prompt_len = prompt_inputs["input_ids"].shape[1]
full_len = full_inputs["input_ids"].shape[1]

# Ensure prompt tokens align before masking the loss
prefix_matches = torch.equal(
    full_inputs["input_ids"][0, :prompt_len],
    prompt_inputs["input_ids"][0]
)

print("Training example:", sid)
print("Images:", len(paths))
print("Prompt tokens:", prompt_len)
print("Total tokens:", full_len)
print("Prompt matches full-input prefix:", prefix_matches)

assert prefix_matches, (
    "Prompt tokens do not align. Stop before training; share this output."
)
assert full_len > prompt_len, "No target tokens found."

# Mask prompt tokens so the loss applies only to the labelled answer
model_inputs = {
    k: v.to(model.device)
    for k, v in full_inputs.items()
    if torch.is_tensor(v)
}

target_labels = model_inputs["input_ids"].clone()
target_labels[:, :prompt_len] = -100

if "attention_mask" in model_inputs:
    target_labels[model_inputs["attention_mask"] == 0] = -100

target_token_count = int((target_labels != -100).sum().item())
print("Target tokens used for loss:", target_token_count)

assert target_token_count > 0

# Test one forward/backward pass without updating the weights
model.config.use_cache = False
model.train()
model.zero_grad(set_to_none=True)
torch.cuda.empty_cache()

model_inputs["labels"] = target_labels

try:
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        outputs = model(**model_inputs)

    loss_value = float(outputs.loss.detach().float().cpu())
    outputs.loss.backward()

    trainable_with_grad = sum(
        1 for p in model.parameters()
        if p.requires_grad and p.grad is not None
    )
    trainable_total = sum(
        1 for p in model.parameters() if p.requires_grad
    )

    print("\nDry-run loss:", loss_value)
    print("Trainable parameters receiving gradients:",
          trainable_with_grad, "/", trainable_total)
    print("Gradient test passed:",
          trainable_with_grad > 0 and loss_value == loss_value)

finally:
    model.zero_grad(set_to_none=True)
    if "outputs" in locals():
        del outputs
    del model_inputs, target_labels, full_inputs, prompt_inputs
    gc.collect()
    torch.cuda.empty_cache()

print("Dry run complete. No optimizer update was performed.")

Training example: seq_06ec0acde37dffa4
Images: 3
Prompt tokens: 4183
Total tokens: 4798
Prompt matches full-input prefix: True
Target tokens used for loss: 615


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.72 GiB. GPU 0 has a total capacity of 14.56 GiB of which 559.81 MiB is free. Including non-PyTorch memory, this process has 14.01 GiB memory in use. Of the allocated memory 11.81 GiB is allocated by PyTorch, and 2.07 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

after failing and running out ofmemeory, lets clean

In [ ]:
import gc
import torch

model.zero_grad(set_to_none=True)

for name in ["outputs", "model_inputs", "target_labels",
             "full_inputs", "prompt_inputs",
             "image_inputs", "video_inputs"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print("Model loaded:", "model" in globals())
print("QLoRA attached:", hasattr(model, "peft_config"))

GPU allocated: 11.77 GB
GPU reserved:  13.86 GB
Model loaded: True
QLoRA attached: True


In [ ]:
import inspect
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

forward_params = inspect.signature(model.forward).parameters

print("Supports logits_to_keep:", "logits_to_keep" in forward_params)
print("Supports num_logits_to_keep:", "num_logits_to_keep" in forward_params)
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Supports logits_to_keep: False
Supports num_logits_to_keep: False
GPU allocated: 11.77 GB
GPU reserved:  13.86 GB


In [ ]:
import inspect

base = model.get_base_model()

print("Base model:", type(base).__name__)
print("Transformer:", type(base.model).__name__)
print("Language-model head:", type(base.lm_head).__name__)

print("\nTransformer forward parameters:")
print(list(inspect.signature(base.model.forward).parameters.keys()))

print("\nLM head weight shape:", tuple(base.lm_head.weight.shape))
print("4-bit quantization:", getattr(base, "is_loaded_in_4bit", False))
print("GPU allocated:", f"{__import__('torch').cuda.memory_allocated()/1024**3:.2f} GB")

Base model: Qwen2_5_VLForConditionalGeneration
Transformer: Qwen2_5_VLModel
Language-model head: Linear

Transformer forward parameters:
['input_ids', 'attention_mask', 'position_ids', 'past_key_values', 'inputs_embeds', 'use_cache', 'pixel_values', 'pixel_values_videos', 'image_grid_thw', 'video_grid_thw', 'mm_token_type_ids', 'second_per_grid_ts', 'mm_encoder_outputs', 'kwargs']

LM head weight shape: (151936, 2048)
4-bit quantization: True
GPU allocated: 11.77 GB


In [ ]:
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info

sid = train_ids[0]
paths = image_paths_by_sequence[sid]

messages = [{
    "role": "user",
    "content": (
        [{"type": "image", "image": str(p)} for p in paths]
        + [{"type": "text", "text": "Transcribe the text on these pages."}]
    )
}]

image_inputs, video_inputs = process_vision_info(messages)

encoded = processor(
    text=[processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

print("Input token shape:", tuple(encoded["input_ids"].shape))
print("Pixel tensor shape:", tuple(encoded["pixel_values"].shape))
print("Image grid:", encoded.get("image_grid_thw"))
print("Configured image size:", processor.image_processor.size)

Input token shape: (1, 4083)
Pixel tensor shape: (16200, 1176)
Image grid: tensor([[ 1, 90, 60],
        [ 1, 90, 60],
        [ 1, 90, 60]])
Configured image size: SizeDict(height=None, width=None, longest_edge=12845056, shortest_edge=3136, max_height=None, max_width=None, min_pixels=None, max_pixels=262144)


In [ ]:
from qwen_vl_utils import process_vision_info
import torch

# Reduce the image pixel budget
processor.image_processor.size["max_pixels"] = 256 * 256

sid = train_ids[0]
paths = image_paths_by_sequence[sid]

messages = [{
    "role": "user",
    "content": (
        [{"type": "image", "image": str(p)} for p in paths]
        + [{"type": "text", "text": "Transcribe the text on these pages."}]
    )
}]

image_inputs, video_inputs = process_vision_info(messages)

encoded = processor(
    text=[processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

print("Input token shape:", tuple(encoded["input_ids"].shape))
print("Pixel tensor shape:", tuple(encoded["pixel_values"].shape))
print("Image grid:", encoded.get("image_grid_thw"))
print("Configured image size:", processor.image_processor.size)

Input token shape: (1, 4083)
Pixel tensor shape: (16200, 1176)
Image grid: tensor([[ 1, 90, 60],
        [ 1, 90, 60],
        [ 1, 90, 60]])
Configured image size: SizeDict(height=None, width=None, longest_edge=12845056, shortest_edge=3136, max_height=None, max_width=None, min_pixels=None, max_pixels=65536)


In [ ]:
import inspect

print(inspect.signature(processor.image_processor.__call__))
print("\nResize implementation:")
print(inspect.getsource(type(processor.image_processor).preprocess)[:5000])

(images: Union[ForwardRef('PIL.Image.Image'), numpy.ndarray, ForwardRef('torch.Tensor'), list['PIL.Image.Image'], list[numpy.ndarray], list['torch.Tensor']], *args, **kwargs: Unpack[transformers.processing_utils.ImagesKwargs]) -> transformers.image_processing_base.BatchFeature

Resize implementation:
    @auto_docstring
    def preprocess(
        self,
        images: ImageInput,
        **kwargs: Unpack[Qwen2VLImageProcessorKwargs],
    ) -> BatchFeature:
        return super().preprocess(images, **kwargs)



we have stopped resizing image cause it kinda gave no results. next is a cleanup cell

In [ ]:
import gc
import sys
import torch

# Release references retained by the failed notebook execution
ip = get_ipython()
if ip is not None:
    result = getattr(ip, "last_execution_result", None)
    if result is not None:
        result.error_in_exec = None
        result.error_before_exec = None

for name in ("last_traceback", "last_value", "last_type"):
    if hasattr(sys, name):
        setattr(sys, name, None)

model.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()

print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print("QLoRA attached:", hasattr(model, "peft_config"))

GPU allocated: 11.77 GB
GPU reserved:  13.86 GB
QLoRA attached: True


In [ ]:
import gc
import sys
import torch

ip = get_ipython()

# Clear references to the failed execution and its traceback
for attr in ("last_value", "last_traceback", "last_type", "last_exc"):
    if hasattr(sys, attr):
        try:
            setattr(sys, attr, None)
        except Exception:
            pass

if ip is not None:
    result = getattr(ip, "last_execution_result", None)
    if result is not None:
        for attr in ("error_in_exec", "error_before_exec", "result"):
            if hasattr(result, attr):
                try:
                    setattr(result, attr, None)
                except Exception:
                    pass

    # Clear cached cell outputs that may retain tensors
    output_cache = ip.user_ns.get("Out")
    if isinstance(output_cache, dict):
        output_cache.clear()

    for name in ("_", "__", "___"):
        ip.user_ns.pop(name, None)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print("Model loaded:", "model" in globals())
print("QLoRA attached:", hasattr(model, "peft_config"))

GPU allocated: 11.77 GB
GPU reserved:  13.86 GB
Model loaded: True
QLoRA attached: True


we restarted becuase cleanup was not working . so let see what survived

In [ ]:
import torch
from pathlib import Path

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

print("\nRequired files:")
for p in [
    "/content/images-20261010T124807Z-1-001.zip",
    "/content/test-20261010T125233Z-1-001.zip",
    "/content/labels (1).jsonl",
    "/content/sequences.json",
    "/content/manga_dataset/train_val_split_seed42.json",
    "/content/baseline_predictions.jsonl",
    "/content/baseline_scores.json",
    "/content/score.py",
]:
    print(("FOUND  " if Path(p).exists() else "MISSING"), p)

GPU available: True
GPU: Tesla T4
GPU allocated: 0.00 GB

Required files:
FOUND   /content/images-20261010T124807Z-1-001.zip
FOUND   /content/test-20261010T125233Z-1-001.zip
FOUND   /content/labels (1).jsonl
FOUND   /content/sequences.json
FOUND   /content/manga_dataset/train_val_split_seed42.json
FOUND   /content/baseline_predictions.jsonl
FOUND   /content/baseline_scores.json
FOUND   /content/score.py


restoring training data and attachinf fresh adapters

In [ ]:
import json
import torch
from pathlib import Path
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# 1. Reload the existing pretrained model in 4-bit
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

# 2. Prepare for QLoRA training
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 3. Restore the verified dataset split and labels
with open("/content/manga_dataset/train_val_split_seed42.json") as f:
    split = json.load(f)

train_ids = split["train_ids"]
validation_ids = split["validation_ids"]

with open("/content/labels (1).jsonl", encoding="utf-8") as f:
    labels = {
        record["sequence_id"]: record
        for record in (json.loads(line) for line in f if line.strip())
    }

# 4. Restore image paths
DATA_ROOT = Path("/content/manga_dataset")
image_paths_by_sequence = {}

for path in DATA_ROOT.rglob("*"):
    if (
        path.is_file()
        and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
        and path.parent.name in set(train_ids) | set(validation_ids)
    ):
        image_paths_by_sequence.setdefault(path.parent.name, []).append(path)

for sid in image_paths_by_sequence:
    image_paths_by_sequence[sid].sort(key=lambda p: p.name)

assert len(train_ids) == 64
assert len(validation_ids) == 16
assert all(sid in labels for sid in train_ids)
assert all(len(image_paths_by_sequence.get(sid, [])) == 3 for sid in train_ids)
assert all(len(image_paths_by_sequence.get(sid, [])) == 3 for sid in validation_ids)

model.eval()

print("\nTraining setup restored.")
print("Training sequences:", len(train_ids))
print("Validation sequences:", len(validation_ids))
print("GPU allocated:", f"{torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("Model loaded:", True)

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

trainable params: 18,576,384 || all params: 3,773,199,360 || trainable%: 0.4923

Training setup restored.
Training sequences: 64
Validation sequences: 16
GPU allocated: 2.90 GB
Model loaded: True


In [ ]:
import gc
import json
import torch
import torch.nn.functional as F
from qwen_vl_utils import process_vision_info

model.train()
model.zero_grad(set_to_none=True)
torch.cuda.empty_cache()

# One training example; never use validation data here
sid = train_ids[0]
record = labels[sid]
paths = image_paths_by_sequence[sid]

instruction = """
Transcribe dialogue, thoughts, narration, and vocalizations from these
three consecutive manga pages in reading order. Exclude sound effects,
credits, signs, and other non-dialogue text.
Return only JSON with exactly three page arrays:
{"pages": [[{"speaker":"character1","text":"..."}], [], []]}
Every line must contain speaker and text. Preserve punctuation and page
boundaries. Keep speaker labels consistent across pages. Use UNKNOWN
when the speaker cannot be identified. Do not add explanations.
"""

user_message = {
    "role": "user",
    "content": (
        [{"type": "image", "image": str(p)} for p in paths]
        + [{"type": "text", "text": instruction}]
    )
}

target = json.dumps({"pages": record["pages"]}, ensure_ascii=False)

messages = [
    user_message,
    {"role": "assistant", "content": [{"type": "text", "text": target}]}
]

# Tokenize full example and prompt separately to identify answer positions
full_text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=False
)
prompt_text = processor.apply_chat_template(
    [user_message], tokenize=False, add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

full_inputs = processor(
    text=[full_text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)
prompt_inputs = processor(
    text=[prompt_text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

prompt_len = prompt_inputs["input_ids"].shape[1]
full_len = full_inputs["input_ids"].shape[1]

assert torch.equal(
    full_inputs["input_ids"][0, :prompt_len],
    prompt_inputs["input_ids"][0]
), "Prompt alignment failed. Stop."

assert full_len > prompt_len, "No answer tokens found."

device = model.device
model_inputs = {
    k: v.to(device)
    for k, v in full_inputs.items()
    if torch.is_tensor(v)
}

input_ids = model_inputs["input_ids"]
target_ids = input_ids[:, prompt_len:]

print("Sequence:", sid)
print("Input tokens:", full_len)
print("Prompt tokens:", prompt_len)
print("Answer tokens:", target_ids.shape[1])

# Call the underlying transformer, not the full causal-LM forward.
# This returns hidden states without creating logits for prompt tokens.
base = model.get_base_model()

with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = base.model(
        **model_inputs,
        use_cache=False,
        return_dict=True
    )

    # Position t-1 predicts target token t
    answer_hidden = outputs.last_hidden_state[:, prompt_len - 1:-1, :]

    assert answer_hidden.shape[1] == target_ids.shape[1], (
        f"Hidden/target length mismatch: {answer_hidden.shape[1]} "
        f"vs {target_ids.shape[1]}"
    )

    # Only answer positions go through the large vocabulary projection
    answer_logits = base.lm_head(answer_hidden)

    print("Answer-only logits shape:", tuple(answer_logits.shape))

    loss = F.cross_entropy(
        answer_logits.float().reshape(-1, answer_logits.shape[-1]),
        target_ids.reshape(-1)
    )

print("Loss:", float(loss.detach().cpu()))
assert torch.isfinite(loss), "Loss is not finite."

loss.backward()

adapter_params = [
    p for p in model.parameters()
    if p.requires_grad
]
params_with_grad = sum(p.grad is not None for p in adapter_params)
finite_grads = all(
    torch.isfinite(p.grad).all().item()
    for p in adapter_params if p.grad is not None
)

print("Adapter parameters receiving gradients:",
      params_with_grad, "/", len(adapter_params))
print("Gradients finite:", finite_grads)
print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# No optimizer step: this was a test only
model.zero_grad(set_to_none=True)
del outputs, answer_hidden, answer_logits, loss
del full_inputs, prompt_inputs, model_inputs
gc.collect()
torch.cuda.empty_cache()

print("Dry run finished. No adapter update was performed.")

Sequence: seq_06ec0acde37dffa4
Input tokens: 4793
Prompt tokens: 4178
Answer tokens: 615


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Answer-only logits shape: (1, 615, 151936)
Loss: 0.3588854670524597
Adapter parameters receiving gradients: 696 / 696
Gradients finite: True
GPU allocated: 3.43 GB
Dry run finished. No adapter update was performed.




> lets start training



In [ ]:
import os
import gc
import json
import time
import torch
import torch.nn.functional as F
from pathlib import Path
from qwen_vl_utils import process_vision_info

# --------------------------------------------------
# 1. Training configuration
# --------------------------------------------------
OUTPUT_DIR = Path("/content/manga_qlora_adapter")
LOG_PATH = Path("/content/manga_qlora_training_log.jsonl")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 1
GRAD_ACCUMULATION = 4
LEARNING_RATE = 1e-4

# Safety checks: use training IDs only
assert len(train_ids) == 64
assert len(validation_ids) == 16
assert set(train_ids).isdisjoint(validation_ids)
assert all(sid in labels for sid in train_ids)

# The model and adapters already exist in memory
assert hasattr(model, "peft_config"), "QLoRA adapters are missing."
model.config.use_cache = False
model.train()
model.zero_grad(set_to_none=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

instruction = """
Transcribe dialogue, thoughts, narration, and vocalizations from these
three consecutive English manga pages in reading order. Exclude sound
effects, credits, signs, and other non-dialogue text.
Return only valid JSON with exactly three page arrays:
{"pages": [[{"speaker":"character1","text":"..."}], [], []]}
Every line must contain speaker and text. Preserve punctuation and page
boundaries. Keep speaker labels consistent across all three pages.
Use UNKNOWN when the speaker cannot be identified. Do not add explanations.
"""

# Deterministic order for this first training run
ordered_train_ids = list(train_ids)
total_examples = len(ordered_train_ids)
optimizer_steps = 0
running_loss = 0.0
started = time.time()

# Start a fresh log for this run
with open(LOG_PATH, "w", encoding="utf-8") as log_file:
    log_file.write(json.dumps({
        "event": "training_started",
        "train_sequences": total_examples,
        "validation_sequences": len(validation_ids),
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "gradient_accumulation": GRAD_ACCUMULATION,
        "loss_method": "answer-token-only logits",
    }) + "\n")

# --------------------------------------------------
# 2. Train on each labelled sequence
# --------------------------------------------------
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for index, sid in enumerate(ordered_train_ids):
        record = labels[sid]
        paths = image_paths_by_sequence[sid]

        assert len(paths) == 3, f"{sid} does not have three images."

        user_message = {
            "role": "user",
            "content": (
                [{"type": "image", "image": str(p)} for p in paths]
                + [{"type": "text", "text": instruction}]
            )
        }

        target = json.dumps(
            {"pages": record["pages"]},
            ensure_ascii=False
        )

        messages = [
            user_message,
            {"role": "assistant", "content": [
                {"type": "text", "text": target}
            ]}
        ]

        # Build the full conversation and prompt-only versions
        full_text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        prompt_text = processor.apply_chat_template(
            [user_message], tokenize=False, add_generation_prompt=True
        )

        image_inputs, video_inputs = process_vision_info(messages)

        full_inputs = processor(
            text=[full_text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        prompt_inputs = processor(
            text=[prompt_text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )

        prompt_len = prompt_inputs["input_ids"].shape[1]
        full_len = full_inputs["input_ids"].shape[1]

        # Ensure answer positions align exactly with the prompt boundary
        if not torch.equal(
            full_inputs["input_ids"][0, :prompt_len],
            prompt_inputs["input_ids"][0]
        ):
            raise RuntimeError(
                f"Prompt alignment failed for {sid}; training stopped."
            )

        device = model.device
        model_inputs = {
            key: value.to(device)
            for key, value in full_inputs.items()
            if torch.is_tensor(value)
        }

        input_ids = model_inputs["input_ids"]
        target_ids = input_ids[:, prompt_len:]

        if target_ids.shape[1] == 0:
            raise RuntimeError(f"No target tokens found for {sid}.")

        # Forward through the transformer without full-sequence vocabulary logits
        base = model.get_base_model()

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = base.model(
                **model_inputs,
                use_cache=False,
                return_dict=True
            )

            # Position t-1 predicts target token t
            answer_hidden = outputs.last_hidden_state[
                :, prompt_len - 1:-1, :
            ]

            if answer_hidden.shape[1] != target_ids.shape[1]:
                raise RuntimeError(
                    f"Loss alignment mismatch for {sid}: "
                    f"{answer_hidden.shape[1]} hidden positions vs "
                    f"{target_ids.shape[1]} targets."
                )

            # Calculate vocabulary logits only for answer positions
            answer_logits = base.lm_head(answer_hidden)

            loss = F.cross_entropy(
                answer_logits.float().reshape(
                    -1, answer_logits.shape[-1]
                ),
                target_ids.reshape(-1)
            )

            # Scale the loss for gradient accumulation
            scaled_loss = loss / GRAD_ACCUMULATION

        loss_value = float(loss.detach().float().cpu())
        scaled_loss.backward()

        running_loss += loss_value
        epoch_loss += loss_value

        # Release per-example activations before processing the next sequence
        del outputs, answer_hidden, answer_logits
        del scaled_loss, loss, target_ids, input_ids, model_inputs
        del full_inputs, prompt_inputs, image_inputs, video_inputs
        del messages, user_message, full_text, prompt_text, target
        gc.collect()

        # Update adapter weights after every four examples
        if (index + 1) % GRAD_ACCUMULATION == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            optimizer_steps += 1

            # Save the latest adapter after each optimizer update
            model.save_pretrained(
                str(OUTPUT_DIR),
                safe_serialization=True
            )

        # Log every example so progress remains inspectable
        log_record = {
            "event": "example_completed",
            "epoch": epoch + 1,
            "index": index + 1,
            "sequence_id": sid,
            "loss": loss_value,
            "optimizer_steps": optimizer_steps,
            "elapsed_minutes": round(
                (time.time() - started) / 60, 2
            ),
        }

        with open(LOG_PATH, "a", encoding="utf-8") as log_file:
            log_file.write(json.dumps(log_record) + "\n")

        if (index + 1) % 4 == 0 or index == 0:
            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Sequence {index + 1}/{total_examples} | "
                f"ID: {sid} | "
                f"Loss: {loss_value:.4f} | "
                f"Optimizer updates: {optimizer_steps} | "
                f"GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB"
            )

    # All 64 examples divide evenly into groups of four
    assert total_examples % GRAD_ACCUMULATION == 0

    average_loss = epoch_loss / total_examples

    # Save the completed epoch and processor configuration
    model.save_pretrained(
        str(OUTPUT_DIR),
        safe_serialization=True
    )
    processor.save_pretrained(str(OUTPUT_DIR))

    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(json.dumps({
            "event": "epoch_completed",
            "epoch": epoch + 1,
            "average_training_loss": average_loss,
            "optimizer_steps": optimizer_steps,
            "elapsed_minutes": round(
                (time.time() - started) / 60, 2
            ),
        }) + "\n")

    print(f"\nEpoch {epoch + 1} complete.")
    print(f"Average training loss: {average_loss:.4f}")

print("\nTRAINING COMPLETE")
print("Optimizer updates:", optimizer_steps)
print("Adapter saved to:", OUTPUT_DIR)
print("Training log saved to:", LOG_PATH)
print(f"Total elapsed minutes: {(time.time() - started)/60:.2f}")

NameError: name 'train_ids' is not defined

damn it got out of memory again. lets see if we had a checkpoint

In [ ]:
import os
import json
import torch

paths = [
    "/content/manga_lora_adapter/trained_checkpoint",
    "/content/manga_qlora_adapter",
    "/content/manga_qlora_training_log.jsonl",
    "/content/manga_training_log.jsonl",
    "/content/baseline_predictions.jsonl",
    "/content/baseline_scores.json",
    "/content/score.py",
]

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("\nFILE STATUS")

for path in paths:
    if os.path.isdir(path):
        files = os.listdir(path)
        print(f"\n{path}/")
        print("  Exists; contents:", files[:15])
    elif os.path.isfile(path):
        print(f"\n{path}")
        if path.endswith(".jsonl") and "training_log" in path:
            with open(path) as f:
                lines = f.readlines()
            print("  Log entries:", len(lines))
            print("  Last entries:", "".join(lines[-3:]))
        else:
            print("  Exists; size:", os.path.getsize(path), "bytes")
    else:
        print(f"\n{path}: NOT FOUND")

GPU: Tesla T4

FILE STATUS

/content/manga_lora_adapter/trained_checkpoint: NOT FOUND

/content/manga_qlora_adapter/
  Exists; contents: []

/content/manga_qlora_training_log.jsonl
  Log entries: 3
  Last entries: {"event": "training_started", "train_sequences": 64, "validation_sequences": 16, "epochs": 1, "learning_rate": 0.0001, "gradient_accumulation": 4, "loss_method": "answer-token-only logits"}
{"event": "example_completed", "epoch": 1, "index": 1, "sequence_id": "seq_06ec0acde37dffa4", "loss": 0.36050689220428467, "optimizer_steps": 0, "elapsed_minutes": 0.61}
{"event": "example_completed", "epoch": 1, "index": 2, "sequence_id": "seq_07838fb26cd1be54", "loss": 0.271785706281662, "optimizer_steps": 0, "elapsed_minutes": 1.15}


/content/manga_training_log.jsonl: NOT FOUND

/content/baseline_predictions.jsonl
  Exists; size: 19566 bytes

/content/baseline_scores.json
  Exists; size: 16728 bytes

/content/score.py
  Exists; size: 13621 bytes


In [ ]:
import inspect
from qwen_vl_utils import process_vision_info

print("Image-processing function:", inspect.signature(process_vision_info))
print("\nGPU memory currently allocated:",
      round(__import__("torch").cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU memory currently reserved:",
      round(__import__("torch").cuda.memory_reserved() / 1024**3, 2), "GB")

Image-processing function: (conversations: Union[List[Dict[str, Any]], List[List[Dict[str, Any]]]], return_video_kwargs: bool = False, return_video_metadata: bool = False, image_patch_size: int = 14) -> Tuple[Optional[List[PIL.Image.Image]], Optional[List[Union[torch.Tensor, List[PIL.Image.Image]]]], Optional[Dict[str, Any]]]

GPU memory currently allocated: 8.01 GB
GPU memory currently reserved: 13.95 GB


have restarted again for one final trainig attempt trying it does not go out of memory

In [ ]:
import os
import torch

required_files = [
    "/content/manga_dataset/train_val_split_seed42.json",
    "/content/labels (1).jsonl",
    "/content/sequences.json",
    "/content/baseline_predictions.jsonl",
    "/content/baseline_scores.json",
    "/content/score.py",
    "/content/manga_qlora_training_log.jsonl",
]

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
print("\nFile verification:")

for path in required_files:
    print(("FOUND  " if os.path.exists(path) else "MISSING"), path)

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

GPU: Tesla T4

File verification:
FOUND   /content/manga_dataset/train_val_split_seed42.json
FOUND   /content/labels (1).jsonl
FOUND   /content/sequences.json
FOUND   /content/baseline_predictions.jsonl
FOUND   /content/baseline_scores.json
FOUND   /content/score.py
FOUND   /content/manga_qlora_training_log.jsonl

GPU memory allocated: 0.0 GB
GPU memory reserved: 0.0 GB


In [ ]:
from PIL import Image
import os
import json

# Load the existing train/validation split
with open("/content/manga_dataset/train_val_split_seed42.json") as f:
    split = json.load(f)

sequence_id = split["train_ids"][0]
image_dir = f"/content/manga_dataset/images-20261010T124807Z-1-001/images/{sequence_id}"

# Resize each page to a maximum image area of 512 × 512 pixels.
# Preserve aspect ratio so the page is not distorted.
MAX_PIXELS = 512 * 512

print("Sequence:", sequence_id)
print("Image directory exists:", os.path.isdir(image_dir))

for filename in sorted(os.listdir(image_dir)):
    if not filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        continue

    path = os.path.join(image_dir, filename)

    with Image.open(path) as image:
        original_size = image.size
        resized = image.copy()
        resized.thumbnail((512, 512), Image.Resampling.LANCZOS)

        print(
            f"{filename}: "
            f"original={original_size}, "
            f"resized={resized.size}, "
            f"pixels={resized.width * resized.height:,}"
        )

Sequence: seq_06ec0acde37dffa4
Image directory exists: True
01.png: original=(843, 1250), resized=(345, 512), pixels=176,640
02.png: original=(848, 1250), resized=(347, 512), pixels=177,664
03.png: original=(833, 1250), resized=(341, 512), pixels=174,592


In [ ]:
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import json
import torch

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)

with open("/content/manga_dataset/train_val_split_seed42.json") as f:
    split = json.load(f)

sequence_id = split["train_ids"][0]
image_dir = (
    f"/content/manga_dataset/images-20261010T124807Z-1-001/"
    f"images/{sequence_id}"
)

# Explicitly resize all three pages in memory.
images = []
for filename in sorted(
    f for f in __import__("os").listdir(image_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
):
    with Image.open(f"{image_dir}/{filename}") as im:
        page = im.convert("RGB")
        page.thumbnail((512, 512), Image.Resampling.LANCZOS)
        images.append(page.copy())

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": page}
        for page in images
    ] + [{
        "type": "text",
        "text": "Read the dialogue on these three manga pages."
    }]
}]

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    ),
    images=image_inputs,
    videos=video_inputs,
    return_tensors="pt"
)

print("Pages processed:", len(images))
print("Pixel tensor shape:", tuple(inputs["pixel_values"].shape))
print("Image grid (T, H, W):", inputs["image_grid_thw"].tolist())
print("Total visual grid cells:", int(inputs["image_grid_thw"][:, 1:].prod(dim=1).sum()))

Pages processed: 3
Pixel tensor shape: (2592, 1176)
Image grid (T, H, W): [[1, 36, 24], [1, 36, 24], [1, 36, 24]]
Total visual grid cells: 2592


In [ ]:
import torch
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("\nModel device:", next(model.parameters()).device)
print("4-bit quantization:", model.is_loaded_in_4bit)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

trainable params: 18,576,384 || all params: 3,773,199,360 || trainable%: 0.4923

Model device: cuda:0
4-bit quantization: True
GPU memory allocated: 2.9 GB


training on one set

In [ ]:
import os
import json
import gc
import torch
import torch.nn.functional as F
from PIL import Image

# --------------------------------------------------
# 1. Load one training example and its ground truth
# --------------------------------------------------
with open("/content/manga_dataset/train_val_split_seed42.json") as f:
    split = json.load(f)

train_id = split["train_ids"][0]

labels_path = "/content/labels (1).jsonl"
labels_by_id = {}

with open(labels_path) as f:
    for line in f:
        item = json.loads(line)
        labels_by_id[item["sequence_id"]] = item

assert train_id in labels_by_id, f"Missing labels for {train_id}"

image_dir = (
    "/content/manga_dataset/"
    "images-20261010T124807Z-1-001/images/"
    + train_id
)

image_files = sorted(
    name for name in os.listdir(image_dir)
    if name.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
)

assert len(image_files) == 3, (
    f"Expected 3 pages, found {len(image_files)}"
)

images = []

for name in image_files:
    with Image.open(os.path.join(image_dir, name)) as im:
        page = im.convert("RGB")
        page.thumbnail((512, 512), Image.Resampling.LANCZOS)
        images.append(page.copy())

target = {
    "sequence_id": train_id,
    "pages": labels_by_id[train_id]["pages"],
}

target_text = json.dumps(target, ensure_ascii=False)

print("Sequence:", train_id)
print("Pages:", len(images))
print("Target characters:", len(target_text))

# --------------------------------------------------
# 2. Build user prompt and assistant target
# --------------------------------------------------
user_message = {
    "role": "user",
    "content": [
        {"type": "image", "image": image}
        for image in images
    ] + [{
        "type": "text",
        "text": (
            "Transcribe the dialogue from these three consecutive "
            "English manga pages in reading order. Preserve page "
            "boundaries, punctuation, and consistent speaker IDs. "
            "Exclude sound effects, credits, signs, and other "
            "non-dialogue text. Return only valid JSON with keys "
            "'sequence_id' and 'pages'. Each page must be an array "
            "of objects containing 'speaker' and 'text'."
        )
    }]
}

full_messages = [
    user_message,
    {"role": "assistant", "content": target_text}
]

prompt_text = processor.apply_chat_template(
    [user_message],
    tokenize=False,
    add_generation_prompt=True,
)

full_text = processor.apply_chat_template(
    full_messages,
    tokenize=False,
    add_generation_prompt=False,
)

# Tokenize the prompt and full example to verify alignment.
prompt_inputs = processor(
    text=[prompt_text],
    images=images,
    return_tensors="pt",
)

prompt_ids = prompt_inputs["input_ids"]
prompt_len = prompt_ids.shape[1]

del prompt_inputs, prompt_ids
gc.collect()
torch.cuda.empty_cache()

full_inputs = processor(
    text=[full_text],
    images=images,
    return_tensors="pt",
)

full_ids = full_inputs["input_ids"]
full_len = full_ids.shape[1]

assert full_len > prompt_len, "No target tokens found"

# Ensure the prompt is an exact token prefix of the full example.
check_prompt = processor(
    text=[prompt_text],
    images=images,
    return_tensors="pt",
)["input_ids"]

assert torch.equal(
    check_prompt,
    full_ids[:, :prompt_len]
), "Prompt/target token alignment failed"

del check_prompt
gc.collect()
torch.cuda.empty_cache()

target_ids = full_ids[:, prompt_len:].to("cuda")

# Move model inputs to the GPU.
model_inputs = {
    key: value.to("cuda") if torch.is_tensor(value) else value
    for key, value in full_inputs.items()
}

# --------------------------------------------------
# 3. Forward + backward pass; NO optimizer update
# --------------------------------------------------
model.train()
model.zero_grad(set_to_none=True)

base = model.get_base_model()

torch.cuda.reset_peak_memory_stats()

try:
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        outputs = base.model(
            **model_inputs,
            use_cache=False,
            return_dict=True,
        )

        # Predict target tokens only, reducing logits memory.
        answer_hidden = outputs.last_hidden_state[
            :, prompt_len - 1:-1, :
        ]

        answer_logits = base.lm_head(answer_hidden)

        assert answer_logits.shape[1] == target_ids.shape[1], (
            f"Logit/target mismatch: {answer_logits.shape[1]} vs "
            f"{target_ids.shape[1]}"
        )

        loss = F.cross_entropy(
            answer_logits.float().reshape(-1, answer_logits.shape[-1]),
            target_ids.reshape(-1),
        )

    loss.backward()

    grad_tensors = [
        p.grad
        for p in model.parameters()
        if p.requires_grad and p.grad is not None
    ]

    finite_gradients = all(
        torch.isfinite(g).all().item()
        for g in grad_tensors
    )

    print("\nDRY RUN RESULT")
    print("Prompt tokens:", prompt_len)
    print("Full sequence tokens:", full_len)
    print("Target tokens:", target_ids.shape[1])
    print("Training loss:", round(loss.item(), 4))
    print("Adapter tensors with gradients:", len(grad_tensors))
    print("Gradients finite:", finite_gradients)
    print(
        "Peak GPU memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB",
    )
    print("Optimizer update performed: NO")

finally:
    model.zero_grad(set_to_none=True)
    del outputs
    gc.collect()
    torch.cuda.empty_cache()

Sequence: seq_06ec0acde37dffa4
Pages: 3
Target characters: 1901


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



DRY RUN RESULT
Prompt tokens: 744
Full sequence tokens: 1378
Target tokens: 634
Training loss: 0.8697
Adapter tensors with gradients: 696
Gradients finite: True
Peak GPU memory: 5.59 GB
Optimizer update performed: NO


In [ ]:
import os
import json
import gc
import time
import torch
from PIL import Image
from torch.utils.checkpoint import checkpoint

# ---------------- CONFIG ----------------
BASE_DIR = "/content/manga_dataset"
IMAGE_ROOT = (
    "/content/manga_dataset/"
    "images-20261010T124807Z-1-001/images"
)
LABELS_PATH = "/content/labels (1).jsonl"
SPLIT_PATH = f"{BASE_DIR}/train_val_split_seed42.json"
OUTPUT_DIR = "/content/manga_qlora_adapter"

MAX_SIDE = 512
LEARNING_RATE = 5e-5
EPOCHS = 1
SAVE_EVERY = 1

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- LOAD DATA ----------------
with open(SPLIT_PATH) as f:
    split = json.load(f)

with open(LABELS_PATH) as f:
    label_rows = [json.loads(line) for line in f if line.strip()]

labels_by_id = {x["sequence_id"]: x for x in label_rows}
train_ids = split["train_ids"]
validation_ids = split["validation_ids"]

assert len(train_ids) == 64, f"Expected 64 train IDs, got {len(train_ids)}"
assert len(validation_ids) == 16
assert not set(train_ids).intersection(validation_ids)
assert all(sid in labels_by_id for sid in train_ids)

print("Training sequences:", len(train_ids))
print("Validation sequences reserved:", len(validation_ids))

# ---------------- HELPERS ----------------
def load_resized_pages(sequence_id):
    folder = os.path.join(IMAGE_ROOT, sequence_id)
    names = sorted(
        n for n in os.listdir(folder)
        if n.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
    )
    if len(names) != 3:
        raise ValueError(f"{sequence_id}: expected 3 pages, found {len(names)}")

    pages = []
    for name in names:
        with Image.open(os.path.join(folder, name)) as im:
            page = im.convert("RGB")
            page.thumbnail((MAX_SIDE, MAX_SIDE), Image.Resampling.LANCZOS)
            pages.append(page.copy())
    return pages

def make_messages(sequence_id, pages):
    target = {
        "sequence_id": sequence_id,
        "pages": labels_by_id[sequence_id]["pages"],
    }
    target_text = json.dumps(target, ensure_ascii=False)

    user_message = {
        "role": "user",
        "content": [
            {"type": "image", "image": page} for page in pages
        ] + [{
            "type": "text",
            "text": (
                "Transcribe dialogue from these three consecutive English "
                "manga pages in reading order. Preserve page boundaries, "
                "punctuation, and consistent speaker IDs. Exclude sound "
                "effects, credits, signs, and other non-dialogue text. "
                "Return only valid JSON with keys sequence_id and pages. "
                "Each page must be an array of objects containing speaker "
                "and text."
            )
        }]
    }
    full_messages = [
        user_message,
        {"role": "assistant", "content": target_text}
    ]

    prompt_text = processor.apply_chat_template(
        [user_message],
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = processor.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return prompt_text, full_text

def prepare_example(sequence_id):
    pages = load_resized_pages(sequence_id)
    prompt_text, full_text = make_messages(sequence_id, pages)

    # Process prompt and full conversation using the same three images.
    prompt_batch = processor(
        text=[prompt_text],
        images=pages,
        return_tensors="pt",
    )
    full_batch = processor(
        text=[full_text],
        images=pages,
        return_tensors="pt",
    )

    prompt_ids = prompt_batch["input_ids"]
    full_ids = full_batch["input_ids"]
    prompt_len = prompt_ids.shape[1]

    # Exact prefix match is required before using this masking scheme.
    if not torch.equal(prompt_ids, full_ids[:, :prompt_len]):
        raise ValueError(
            f"{sequence_id}: prompt tokens do not match full-input prefix"
        )
    if full_ids.shape[1] <= prompt_len:
        raise ValueError(f"{sequence_id}: target has no tokens")

    # Labels use standard causal LM loss. Prompt positions are ignored.
    labels = full_ids.clone()
    labels[:, :prompt_len] = -100

    # Move tensors to model's device.
    device = next(model.parameters()).device
    batch = {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in full_batch.items()
    }
    labels = labels.to(device)

    del prompt_batch, full_batch, prompt_ids, full_ids
    return batch, labels

# ---------------- TRAIN ----------------
model.config.use_cache = False
model.train()

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

log_path = f"{OUTPUT_DIR}/training_log.jsonl"
optimizer_steps = 0
start_time = time.time()

with open(log_path, "a") as log:
    log.write(json.dumps({
        "event": "training_started",
        "train_sequences": len(train_ids),
        "validation_sequences": len(validation_ids),
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "image_max_side": MAX_SIDE,
        "loss_method": "standard causal LM loss with prompt masking",
    }) + "\n")

    for epoch in range(EPOCHS):
        # Deterministic order makes this run easier to reproduce.
        for index, sequence_id in enumerate(train_ids, start=1):
            optimizer.zero_grad(set_to_none=True)
            gc.collect()
            torch.cuda.empty_cache()

            try:
                batch, labels = prepare_example(sequence_id)

                with torch.autocast("cuda", dtype=torch.float16):
                    outputs = model(
                        **batch,
                        labels=labels,
                        use_cache=False,
                    )
                    loss = outputs.loss

                if not torch.isfinite(loss).item():
                    raise ValueError(
                        f"Non-finite loss for {sequence_id}: {loss.item()}"
                    )

                loss.backward()

                grads = [
                    p.grad for p in model.parameters()
                    if p.requires_grad and p.grad is not None
                ]
                if not grads or not all(
                    torch.isfinite(g).all().item() for g in grads
                ):
                    raise ValueError(
                        f"Missing or non-finite adapter gradients: {sequence_id}"
                    )

                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    max_norm=1.0,
                )

                optimizer.step()
                optimizer_steps += 1

                record = {
                    "event": "example_completed",
                    "epoch": epoch + 1,
                    "index": index,
                    "sequence_id": sequence_id,
                    "loss": float(loss.detach().item()),
                    "optimizer_steps": optimizer_steps,
                    "elapsed_minutes": round(
                        (time.time() - start_time) / 60, 2
                    ),
                    "peak_gpu_memory_gb": round(
                        torch.cuda.max_memory_allocated() / 1024**3, 2
                    ),
                }
                log.write(json.dumps(record) + "\n")
                log.flush()

                print(
                    f"[{index}/{len(train_ids)}] "
                    f"{sequence_id} | loss={record['loss']:.4f} | "
                    f"updates={optimizer_steps} | "
                    f"peak={record['peak_gpu_memory_gb']:.2f} GB"
                )

                # Save after every successful optimizer update.
                if optimizer_steps % SAVE_EVERY == 0:
                    model.save_pretrained(OUTPUT_DIR)
                    processor.save_pretrained(OUTPUT_DIR)

                    with open(f"{OUTPUT_DIR}/training_state.json", "w") as sf:
                        json.dump({
                            "epoch": epoch + 1,
                            "last_train_index": index,
                            "optimizer_steps": optimizer_steps,
                            "last_sequence_id": sequence_id,
                        }, sf, indent=2)

                del outputs, loss, batch, labels, grads
                gc.collect()
                torch.cuda.empty_cache()

            except Exception as exc:
                log.write(json.dumps({
                    "event": "training_error",
                    "epoch": epoch + 1,
                    "index": index,
                    "sequence_id": sequence_id,
                    "error": repr(exc),
                    "optimizer_steps": optimizer_steps,
                }) + "\n")
                log.flush()
                raise

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

with open(f"{OUTPUT_DIR}/training_state.json", "w") as f:
    json.dump({
        "completed": True,
        "optimizer_steps": optimizer_steps,
        "epochs": EPOCHS,
    }, f, indent=2)

print("\nTRAINING COMPLETE")
print("Optimizer updates:", optimizer_steps)
print("Adapter directory:", OUTPUT_DIR)
print("Training log:", log_path)

Training sequences: 64
Validation sequences reserved: 16


OutOfMemoryError: CUDA out of memory. Tried to allocate 34.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 21.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.08 GiB is allocated by PyTorch, and 328.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
import os
import json

adapter_dir = "/content/manga_qlora_adapter"

print("Adapter directory exists:", os.path.isdir(adapter_dir))

for filename in sorted(os.listdir(adapter_dir)):
    path = os.path.join(adapter_dir, filename)
    if os.path.isfile(path):
        print(f"{filename}: {os.path.getsize(path):,} bytes")

config_path = os.path.join(adapter_dir, "adapter_config.json")

if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    print("\nBase model:", config.get("base_model_name_or_path"))
    print("LoRA rank:", config.get("r"))
    print("Target modules:", config.get("target_modules"))
else:
    print("\nWARNING: adapter_config.json is missing.")

Adapter directory exists: True
README.md: 5,208 bytes
adapter_config.json: 1,182 bytes
adapter_model.safetensors: 74,405,904 bytes
chat_template.jinja: 1,017 bytes
processor_config.json: 1,294 bytes
tokenizer.json: 11,421,892 bytes
tokenizer_config.json: 738 bytes
training_log.jsonl: 7,529 bytes
training_state.json: 113 bytes

Base model: Qwen/Qwen2.5-VL-3B-Instruct
LoRA rank: 8
Target modules: ['gate_proj', 'down_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj', 'up_proj']


This is to not compare apples to oranges.

In [ ]:
import os
import json
import re
import torch
import subprocess
from contextlib import nullcontext
from PIL import Image

# ---------- Paths and settings ----------
ROOT = "/content/manga_dataset"
IMAGE_ROOT = (
    ROOT + "/images-20261010T124807Z-1-001/images"
)
LABELS_PATH = "/content/labels (1).jsonl"
SPLIT_PATH = ROOT + "/train_val_split_seed42.json"
SCORE_SCRIPT = "/content/score.py"
OUT = "/content/manga_evaluation"

os.makedirs(OUT, exist_ok=True)

# ---------- Load labels and validation split ----------
with open(SPLIT_PATH) as f:
    split = json.load(f)

with open(LABELS_PATH) as f:
    label_rows = [json.loads(line) for line in f if line.strip()]

labels_by_id = {x["sequence_id"]: x for x in label_rows}
val_ids = split["validation_ids"]

assert len(val_ids) == 16
assert all(sid in labels_by_id for sid in val_ids)

references_path = OUT + "/validation_references.jsonl"

with open(references_path, "w") as f:
    for sid in val_ids:
        ref = {
            "sequence_id": sid,
            "pages": labels_by_id[sid]["pages"],
        }
        f.write(json.dumps(ref, ensure_ascii=False) + "\n")

# ---------- Resize validation images exactly as in training ----------
def load_pages(sid):
    folder = os.path.join(IMAGE_ROOT, sid)
    names = sorted(
        n for n in os.listdir(folder)
        if n.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
    )
    assert len(names) == 3, f"{sid}: expected 3 pages"

    pages = []
    for name in names:
        with Image.open(os.path.join(folder, name)) as im:
            page = im.convert("RGB")
            page.thumbnail((512, 512), Image.Resampling.LANCZOS)
            pages.append(page.copy())
    return pages

PROMPT = (
    "Transcribe the dialogue from these three consecutive English "
    "manga pages in reading order. Preserve page boundaries, "
    "punctuation, and consistent speaker IDs. Exclude sound effects, "
    "credits, signs, and other non-dialogue text. Return only valid "
    "JSON with keys 'sequence_id' and 'pages'. Each page must be an "
    "array of objects containing 'speaker' and 'text'."
)

def parse_json(text):
    # Try a complete JSON object, including one inside a code fence.
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except Exception:
            pass

    # Page-level fallback may return a JSON array.
    start = text.find("[")
    end = text.rfind("]")
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except Exception:
            pass
    return None

def clean_page(page):
    if not isinstance(page, list):
        return []
    return [
        {"speaker": str(x["speaker"]), "text": str(x["text"])}
        for x in page
        if isinstance(x, dict) and "speaker" in x and "text" in x
    ]

def generate_text(messages, use_adapter):
    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    pages = [
        item["image"] for item in messages[0]["content"]
        if item.get("type") == "image"
    ]

    inputs = processor(
        text=[prompt_text],
        images=pages,
        return_tensors="pt",
    )
    device = next(model.parameters()).device
    inputs = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    context = nullcontext() if use_adapter else model.disable_adapter()

    with context, torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.float16):
            output = model.generate(
                **inputs,
                max_new_tokens=900,
                do_sample=False,
                use_cache=True,
            )

    prompt_length = inputs["input_ids"].shape[1]
    generated = output[:, prompt_length:]
    result = processor.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    del inputs, output, generated
    return result

def run_one(sid, use_adapter):
    pages = load_pages(sid)
    user_message = {
        "role": "user",
        "content": [
            *[{"type": "image", "image": p} for p in pages],
            {"type": "text", "text": PROMPT},
        ],
    }

    raw = generate_text([user_message], use_adapter)
    parsed = parse_json(raw)

    # If sequence-level JSON is malformed, try each page individually.
    if not isinstance(parsed, dict) or not isinstance(parsed.get("pages"), list):
        recovered = []
        for page in pages:
            page_message = {
                "role": "user",
                "content": [
                    {"type": "image", "image": page},
                    {"type": "text", "text": (
                        "Transcribe only dialogue from this manga page. "
                        "Return a JSON array of objects with speaker and text. "
                        "Exclude sound effects, credits, and signs."
                    )},
                ],
            }
            page_raw = generate_text([page_message], use_adapter)
            page_parsed = parse_json(page_raw)
            if isinstance(page_parsed, dict):
                page_parsed = page_parsed.get("dialogue", page_parsed.get("pages"))
            recovered.append(clean_page(page_parsed))

        result = {"sequence_id": sid, "pages": recovered}
        return result, raw, True

    predicted_pages = parsed["pages"][:3]
    while len(predicted_pages) < 3:
        predicted_pages.append([])

    result = {
        "sequence_id": sid,
        "pages": [clean_page(p) for p in predicted_pages],
    }
    return result, raw, False

# ---------- Generate both conditions ----------
model.eval()
model.config.use_cache = True

conditions = [
    ("pretrained_resized", False),
    ("finetuned_resized", True),
]

for condition, use_adapter in conditions:
    pred_path = f"{OUT}/{condition}_predictions.jsonl"
    raw_path = f"{OUT}/{condition}_raw.jsonl"
    print(f"\nEvaluating: {condition}")

    with open(pred_path, "w") as pred_file, open(raw_path, "w") as raw_file:
        for i, sid in enumerate(val_ids, start=1):
            prediction, raw, used_fallback = run_one(sid, use_adapter)
            pred_file.write(json.dumps(prediction, ensure_ascii=False) + "\n")
            raw_file.write(json.dumps({
                "sequence_id": sid,
                "raw_output": raw,
                "page_fallback_used": used_fallback,
            }, ensure_ascii=False) + "\n")
            pred_file.flush()
            raw_file.flush()
            print(
                f"[{i}/16] {sid} | "
                f"page fallback: {used_fallback}"
            )

    score_path = f"{OUT}/{condition}_scores.json"
    result = subprocess.run(
        [
            "python", SCORE_SCRIPT,
            "--references", references_path,
            "--predictions", pred_path,
            "--output", score_path,
        ],
        capture_output=True,
        text=True,
    )

    print(result.stdout[-3000:])
    if result.returncode != 0:
        print("SCORER ERROR:")
        print(result.stderr[-3000:])
        raise RuntimeError(f"Scoring failed for {condition}")

    print("Saved predictions:", pred_path)
    print("Saved scores:", score_path)

print("\nBoth evaluation runs finished.")
print("Results directory:", OUT)


Evaluating: pretrained_resized
[1/16] seq_032cbb4dc8686963 | page fallback: False
[2/16] seq_2022b8c0e1f2d3c0 | page fallback: False
[3/16] seq_2e4b837ead6b88ae | page fallback: False
[4/16] seq_2fae297a18af2a57 | page fallback: False


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[5/16] seq_44b9a316b2919cfe | page fallback: True
[6/16] seq_535e0754d805105b | page fallback: False
[7/16] seq_5e4a9d63617d62bf | page fallback: False
[8/16] seq_5f9ee35846b7e635 | page fallback: False
[9/16] seq_888a2728b36f1745 | page fallback: False
[10/16] seq_9032e5551cf1a9d2 | page fallback: False
[11/16] seq_b8b5563330fdc89d | page fallback: True
[12/16] seq_c2de6cb4b3c91103 | page fallback: True
[13/16] seq_d62d309820d28efa | page fallback: False
[14/16] seq_dce8be23c3281228 | page fallback: False
[15/16] seq_dcebdbbf614a9b4d | page fallback: True
[16/16] seq_f492843aa3a0240c | page fallback: False
{
  "text_order_score": 0.07228224728720807,
  "balanced_joint_f1": 0.018794241213376788,
  "speaker_accuracy_on_matched": 0.12671743220506188,
  "joint_f1": 0.04379372700687336,
  "matched_token_coverage": 0.0645245940976689
}

Saved predictions: /content/manga_evaluation/pretrained_resized_predictions.jsonl
Saved scores: /content/manga_evaluation/pretrained_resized_scores.json

Ev

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[1/16] seq_032cbb4dc8686963 | page fallback: True
[2/16] seq_2022b8c0e1f2d3c0 | page fallback: True
[3/16] seq_2e4b837ead6b88ae | page fallback: True
[4/16] seq_2fae297a18af2a57 | page fallback: False
[5/16] seq_44b9a316b2919cfe | page fallback: True
[6/16] seq_535e0754d805105b | page fallback: False
[7/16] seq_5e4a9d63617d62bf | page fallback: False
[8/16] seq_5f9ee35846b7e635 | page fallback: False
[9/16] seq_888a2728b36f1745 | page fallback: True
[10/16] seq_9032e5551cf1a9d2 | page fallback: False
[11/16] seq_b8b5563330fdc89d | page fallback: True
[12/16] seq_c2de6cb4b3c91103 | page fallback: True
[13/16] seq_d62d309820d28efa | page fallback: True
[14/16] seq_dce8be23c3281228 | page fallback: True
[15/16] seq_dcebdbbf614a9b4d | page fallback: True
[16/16] seq_f492843aa3a0240c | page fallback: False
{
  "text_order_score": 0.24807965741772134,
  "balanced_joint_f1": 0.08902958840412144,
  "speaker_accuracy_on_matched": 0.5690502543273492,
  "joint_f1": 0.16967379018161421,
  "matched

In [ ]:
import os, json, zipfile

split_path = "/content/manga_dataset/train_val_split_seed42.json"
sequences_path = "/content/sequences.json"
test_zip = "/content/test-20261010T125233Z-1-001.zip"
adapter_dir = "/content/manga_qlora_adapter"
submission_path = "/content/sample_submission.jsonl"

# 1. Check the trained adapter
print("ADAPTER")
print("Exists:", os.path.isdir(adapter_dir))
if os.path.isdir(adapter_dir):
    print("Files:", sorted(os.listdir(adapter_dir)))

# 2. Check the train/validation/test split
print("\nDATA SPLIT")
with open(split_path, "r") as f:
    split = json.load(f)

print("Split keys:", list(split.keys()))
for key, value in split.items():
    if isinstance(value, list):
        print(f"{key}: {len(value)} sequences")

test_ids = split.get("test_ids", split.get("test", []))
print("Test sequence count:", len(test_ids))
print("First test IDs:", test_ids[:3])

# 3. Inspect sequence metadata
print("\nSEQUENCE METADATA")
with open(sequences_path, "r") as f:
    sequence_data = json.load(f)

if isinstance(sequence_data, dict):
    print("Top-level keys:", list(sequence_data.keys())[:15])
elif isinstance(sequence_data, list):
    print("Total sequence records:", len(sequence_data))
    print("First record:", sequence_data[0] if sequence_data else "EMPTY")

# 4. Inspect the test archive without changing it
print("\nTEST ARCHIVE")
print("Exists:", os.path.isfile(test_zip))
if os.path.isfile(test_zip):
    with zipfile.ZipFile(test_zip) as z:
        names = z.namelist()
        print("Archive entries:", len(names))
        print("First 15 entries:", names[:15])

# 5. Inspect the submission template
print("\nSUBMISSION TEMPLATE")
print("Exists:", os.path.isfile(submission_path))
if os.path.isfile(submission_path):
    with open(submission_path, "r") as f:
        for _, line in zip(range(2), f):
            print(line.rstrip())

ADAPTER
Exists: True
Files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json', 'training_log.jsonl', 'training_state.json']

DATA SPLIT
Split keys: ['seed', 'train_ids', 'validation_ids']
train_ids: 64 sequences
validation_ids: 16 sequences
Test sequence count: 0
First test IDs: []

SEQUENCE METADATA
Total sequence records: 95
First record: {'sequence_id': 'seq_2032620aa4e4ac7f', 'split': 'development', 'images': ['development/images/seq_2032620aa4e4ac7f/01.png', 'development/images/seq_2032620aa4e4ac7f/02.png', 'development/images/seq_2032620aa4e4ac7f/03.png']}

TEST ARCHIVE
Exists: True
Archive entries: 45
First 15 entries: ['test/images/seq_6aa313a0c0fa1c39/01.png', 'test/images/seq_e04ef3d20fd0baa8/03.png', 'test/images/seq_e04ef3d20fd0baa8/02.png', 'test/images/seq_75aeca6c7131544b/02.png', 'test/images/seq_c5622aca7487a661/02.png', 'test/images/seq_75aeca6c7131544b/01.png', '

In [ ]:
import os
import json
import zipfile
from pathlib import Path
from collections import defaultdict

# Paths
test_zip = "/content/test-20261010T125233Z-1-001.zip"
extract_dir = "/content/manga_test"
output_json = "/content/manga_test_sequences.json"

# Extract test images
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(test_zip, "r") as z:
    z.extractall(extract_dir)

# Group the images by sequence ID
image_root = Path(extract_dir) / "test" / "images"
grouped = defaultdict(list)

for image_path in image_root.rglob("*.png"):
    sequence_id = image_path.parent.name
    grouped[sequence_id].append(str(image_path))

# Sort pages into reading order
test_sequences = []

for sequence_id in sorted(grouped):
    images = sorted(
        grouped[sequence_id],
        key=lambda p: Path(p).name
    )

    test_sequences.append({
        "sequence_id": sequence_id,
        "images": images
    })

# Validate the dataset
print("Number of test sequences:", len(test_sequences))

invalid = [
    s for s in test_sequences
    if len(s["images"]) != 3
]

print("Sequences without exactly 3 pages:", len(invalid))

if invalid:
    for s in invalid:
        print(s["sequence_id"], "has", len(s["images"]), "pages")
    raise ValueError("Fix the test sequence structure before inference.")

# Save the test manifest
with open(output_json, "w") as f:
    json.dump(test_sequences, f, indent=2)

print("\nSaved test manifest:", output_json)
print("\nFirst test sequence:")
print(json.dumps(test_sequences[0], indent=2))

Number of test sequences: 15
Sequences without exactly 3 pages: 0

Saved test manifest: /content/manga_test_sequences.json

First test sequence:
{
  "sequence_id": "seq_14469aeb38963085",
  "images": [
    "/content/manga_test/test/images/seq_14469aeb38963085/01.png",
    "/content/manga_test/test/images/seq_14469aeb38963085/02.png",
    "/content/manga_test/test/images/seq_14469aeb38963085/03.png"
  ]
}


In [ ]:
import os
import json
import re
import gc
import torch
from PIL import Image
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)
from peft import PeftModel

# --------------------------------------------------
# 1. Paths and settings
# --------------------------------------------------
BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_DIR = "/content/manga_qlora_adapter"
MANIFEST = "/content/manga_test_sequences.json"

OUT_DIR = "/content/manga_final_submission"
os.makedirs(OUT_DIR, exist_ok=True)

SUBMISSION = os.path.join(OUT_DIR, "test_predictions.jsonl")
RAW_OUTPUTS = os.path.join(OUT_DIR, "raw_model_outputs.jsonl")
PROGRESS = os.path.join(OUT_DIR, "progress.jsonl")

assert os.path.isdir(ADAPTER_DIR), "Saved adapter not found."
assert os.path.isfile(MANIFEST), "Test manifest not found."
assert torch.cuda.is_available(), "Enable a GPU in Colab first."

with open(MANIFEST, "r") as f:
    test_sequences = json.load(f)

assert len(test_sequences) == 15, (
    f"Expected 15 test sequences; found {len(test_sequences)}."
)

# --------------------------------------------------
# 2. Load the fine-tuned model
# --------------------------------------------------
if not (
    "model" in globals()
    and "processor" in globals()
    and hasattr(model, "peft_config")
):
    # Release any previously loaded model if necessary.
    if "model" in globals():
        del model
        gc.collect()
        torch.cuda.empty_cache()

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    processor = AutoProcessor.from_pretrained(BASE_MODEL)
    processor.image_processor.max_pixels = 65536

    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    model = PeftModel.from_pretrained(
        base,
        ADAPTER_DIR,
        is_trainable=False,
    )
    del base
    gc.collect()
    torch.cuda.empty_cache()

# Ensure the fine-tuned adapter is active.
if hasattr(model, "peft_config"):
    if "default" in model.peft_config:
        model.set_adapter("default")

model.eval()
print("Model loaded and ready.")
print("Adapter:", ADAPTER_DIR)
print("Test sequences:", len(test_sequences))

# --------------------------------------------------
# 3. Helpers for image loading and JSON parsing
# --------------------------------------------------
def load_pages(paths):
    pages = []
    for path in paths:
        with Image.open(path) as im:
            im = im.convert("RGB")
            im.thumbnail((512, 512), Image.Resampling.LANCZOS)
            pages.append(im.copy())
    return pages


def parse_json_output(text):
    """Parse JSON even if the model adds a little extra text."""
    text = text.strip()

    # Try the complete response first.
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try the first JSON object.
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass

    # Try a JSON array, useful for single-page fallback output.
    start, end = text.find("["), text.rfind("]")
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass

    return None


def clean_dialogue(items):
    """Keep only the required speaker/text fields."""
    if not isinstance(items, list):
        return []

    cleaned = []
    for item in items:
        if not isinstance(item, dict):
            continue

        text = item.get("text")
        speaker = item.get("speaker", "UNKNOWN")

        if not isinstance(text, str) or not text.strip():
            continue

        if not isinstance(speaker, str) or not speaker.strip():
            speaker = "UNKNOWN"

        cleaned.append({
            "speaker": speaker.strip(),
            "text": text.strip(),
        })

    return cleaned


def generate_text(prompt, images, max_new_tokens):
    inputs = processor(
        text=[prompt],
        images=images,
        return_tensors="pt",
    )
    inputs = {
        key: value.to(model.device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    # Decode only newly generated tokens, not the prompt.
    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = generated[:, prompt_length:]
    result = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    del inputs, generated, new_tokens
    return result.strip()


# --------------------------------------------------
# 4. Process all 15 test sequences
# --------------------------------------------------
sequence_prompt = (
    "Transcribe the dialogue from these three consecutive English manga "
    "pages in reading order. Preserve page boundaries, punctuation, and "
    "consistent speaker IDs. Exclude sound effects, credits, signs, and "
    "other non-dialogue text. Return only valid JSON with keys "
    "'sequence_id' and 'pages'. Each page must be an array of objects "
    "containing 'speaker' and 'text'."
)

page_prompt = (
    "Transcribe only the dialogue from this English manga page in reading "
    "order. Preserve punctuation. Exclude sound effects, credits, signs, "
    "and other non-dialogue text. Return only a valid JSON array of "
    "objects containing 'speaker' and 'text'."
)

# Read existing progress so an interrupted run can be resumed.
completed = {}
if os.path.isfile(PROGRESS):
    with open(PROGRESS, "r") as f:
        for line in f:
            try:
                row = json.loads(line)
                completed[row["sequence_id"]] = row
            except (json.JSONDecodeError, KeyError):
                continue

# Start fresh output files, then write all completed records in order.
raw_by_id = {}
if os.path.isfile(RAW_OUTPUTS):
    with open(RAW_OUTPUTS, "r") as f:
        for line in f:
            try:
                row = json.loads(line)
                raw_by_id[row["sequence_id"]] = row
            except (json.JSONDecodeError, KeyError):
                continue

for seq in tqdm(test_sequences, desc="Test inference"):
    sid = seq["sequence_id"]

    if sid in completed and sid in raw_by_id:
        continue

    pages = load_pages(seq["images"])
    raw_sequence = ""
    fallback_used = False
    error = None

    try:
        raw_sequence = generate_text(
            sequence_prompt, pages, max_new_tokens=900
        )
        parsed = parse_json_output(raw_sequence)

        predicted_pages = (
            parsed.get("pages") if isinstance(parsed, dict) else None
        )

        # A valid sequence prediction must contain exactly 3 page lists.
        valid = (
            isinstance(predicted_pages, list)
            and len(predicted_pages) == 3
            and all(isinstance(p, list) for p in predicted_pages)
        )

        if valid:
            predicted_pages = [
                clean_dialogue(page) for page in predicted_pages
            ]
        else:
            # Retry one page at a time if sequence-level JSON failed.
            fallback_used = True
            predicted_pages = []

            for page_image in pages:
                raw_page = generate_text(
                    page_prompt, [page_image], max_new_tokens=350
                )
                parsed_page = parse_json_output(raw_page)

                if isinstance(parsed_page, dict):
                    parsed_page = parsed_page.get(
                        "dialogue", parsed_page.get("lines", [])
                    )

                predicted_pages.append(clean_dialogue(parsed_page))

        assert len(predicted_pages) == 3

    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        # Preserve the sequence and page boundaries even on failure.
        predicted_pages = [[], [], []]
        print(f"\nWARNING: {sid}: {error}")

    result = {
        "sequence_id": sid,
        "pages": predicted_pages,
    }

    raw_record = {
        "sequence_id": sid,
        "raw_sequence_output": raw_sequence,
        "fallback_used": fallback_used,
        "error": error,
        "predictions": predicted_pages,
    }

    completed[sid] = result
    raw_by_id[sid] = raw_record

    # Checkpoint after every sequence.
    with open(PROGRESS, "w") as f:
        for item in test_sequences:
            if item["sequence_id"] in completed:
                f.write(json.dumps(
                    completed[item["sequence_id"]],
                    ensure_ascii=False
                ) + "\n")

    with open(RAW_OUTPUTS, "w") as f:
        for item in test_sequences:
            if item["sequence_id"] in raw_by_id:
                f.write(json.dumps(
                    raw_by_id[item["sequence_id"]],
                    ensure_ascii=False
                ) + "\n")

# --------------------------------------------------
# 5. Write and validate final submission
# --------------------------------------------------
with open(SUBMISSION, "w") as f:
    for seq in test_sequences:
        sid = seq["sequence_id"]
        row = completed[sid]
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(SUBMISSION, "r") as f:
    final_rows = [json.loads(line) for line in f if line.strip()]

assert len(final_rows) == 15
assert len({row["sequence_id"] for row in final_rows}) == 15

for row in final_rows:
    assert set(row.keys()) == {"sequence_id", "pages"}
    assert len(row["pages"]) == 3
    for page in row["pages"]:
        for dialogue in page:
            assert set(dialogue.keys()) == {"speaker", "text"}

print("\nINFERENCE COMPLETE")
print("Sequences:", len(final_rows))
print("Pages:", sum(len(row["pages"]) for row in final_rows))
print("Sequences with fallback:", sum(
    1 for row in raw_by_id.values() if row.get("fallback_used")
))
print("Sequences with errors:", sum(
    1 for row in raw_by_id.values() if row.get("error")
))
print("\nSubmission:", SUBMISSION)
print("Raw outputs:", RAW_OUTPUTS)
print("Progress checkpoint:", PROGRESS)
print("\nFirst prediction:")
print(json.dumps(final_rows[0], indent=2, ensure_ascii=False))

Model loaded and ready.
Adapter: /content/manga_qlora_adapter
Test sequences: 15


Test inference:   0%|          | 0/15 [00:00<?, ?it/s]

















INFERENCE COMPLETE
Sequences: 15
Pages: 45
Sequences with fallback: 0
Sequences with errors: 15

Submission: /content/manga_final_submission/test_predictions.jsonl
Raw outputs: /content/manga_final_submission/raw_model_outputs.jsonl
Progress checkpoint: /content/manga_final_submission/progress.jsonl

First prediction:
{
  "sequence_id": "seq_14469aeb38963085",
  "pages": [
    [],
    [],
    []
  ]
}


In [ ]:
import json
import torch
from PIL import Image

# Load one test sequence
with open("/content/manga_test_sequences.json", "r") as f:
    test_sequences = json.load(f)

seq = test_sequences[0]
pages = []

for path in seq["images"]:
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((512, 512), Image.Resampling.LANCZOS)
        pages.append(im.copy())

prompt = (
    "Transcribe the dialogue from these three consecutive English manga "
    "pages in reading order. Preserve page boundaries, punctuation, and "
    "consistent speaker IDs. Exclude sound effects, credits, signs, and "
    "other non-dialogue text. Return only valid JSON with keys "
    "'sequence_id' and 'pages'. Each page must be an array of objects "
    "containing 'speaker' and 'text'."
)

# Build a Qwen-compatible conversation with explicit image markers.
content = [{"type": "image", "image": image} for image in pages]
content.append({"type": "text", "text": prompt})

messages = [{"role": "user", "content": content}]

formatted_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# The formatted prompt now contains the image markers expected by Qwen.
inputs = processor(
    text=[formatted_prompt],
    images=pages,
    return_tensors="pt",
)

inputs = {
    k: v.to(model.device) if torch.is_tensor(v) else v
    for k, v in inputs.items()
}

print("Sequence:", seq["sequence_id"])
print("Input tokens:", inputs["input_ids"].shape[1])
print("Image tensor shape:", tuple(inputs["pixel_values"].shape))

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=900,
        do_sample=False,
        use_cache=True,
        pad_token_id=processor.tokenizer.pad_token_id,
    )

new_tokens = output[:, inputs["input_ids"].shape[1]:]
raw_text = processor.batch_decode(
    new_tokens,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

print("\nRAW MODEL RESPONSE:\n")
print(raw_text)

# Check whether the model returned usable JSON.
try:
    parsed = json.loads(raw_text)
except json.JSONDecodeError:
    parsed = None

valid = (
    isinstance(parsed, dict)
    and isinstance(parsed.get("pages"), list)
    and len(parsed["pages"]) == 3
    and all(isinstance(page, list) for page in parsed["pages"])
)

print("\nValid three-page JSON:", valid)

if valid:
    print("Dialogue lines per page:",
          [len(page) for page in parsed["pages"]])

Sequence: seq_14469aeb38963085
Input tokens: 798
Image tensor shape: (2808, 1176)

RAW MODEL RESPONSE:

{"sequence_id": "seq_9854076321e9d9a", "pages": [[{"speaker": "character1", "text": "IT LOOKS LIKE THE POLICE IS GOING TO SUCCEED HER OUT OF THIS PLACE."}, {"speaker": "character1", "text": "THIS IS BAD."}, {"speaker": "character1", "text": "THEN JUST WILL PROBABLY LEAVE."}, {"speaker": "character1", "text": "I GET THE FEELING ABOUT SAIKO..."}, {"speaker": "character1", "text": "THAT YOU WILL BE SOFT ON HER..."}, {"speaker": "character1", "text": "YOU GOT SOME REASON FOR THAT?"}, {"speaker": "character1", "text": "THEY WILL BE SOFT ON HER..."}, {"speaker": "character1", "text": "SO SOFT..."}, {"speaker": "character1", "text": "BUT..."}, {"speaker": "character1", "text": "I NEED TO MENTOR SAIKO CHAN BECAUSE OF MY POSITION."}, {"speaker": "character1", "text": "HERE'S HOW I PLAN TO DO THAT."}], [{"speaker": "character1", "text": "WITH THE SUPPORT OF HER MOTHER, SAICHI STARTED SAIKO BAR

In [ ]:
import os
import json
import torch
from PIL import Image
from tqdm.auto import tqdm

OUT_DIR = "/content/manga_final_submission_v2"
os.makedirs(OUT_DIR, exist_ok=True)

SUBMISSION = os.path.join(OUT_DIR, "test_predictions.jsonl")
RAW_OUTPUTS = os.path.join(OUT_DIR, "raw_model_outputs.jsonl")

with open("/content/manga_test_sequences.json", "r") as f:
    test_sequences = json.load(f)

assert len(test_sequences) == 15
assert "model" in globals() and "processor" in globals()
assert hasattr(model, "peft_config"), "The fine-tuned adapter is not loaded."

model.eval()

def load_pages(paths):
    result = []
    for path in paths:
        with Image.open(path) as im:
            im = im.convert("RGB")
            im.thumbnail((512, 512), Image.Resampling.LANCZOS)
            result.append(im.copy())
    return result

def generate_with_images(prompt, images, max_new_tokens):
    # Use the Qwen chat template with explicit image markers.
    content = [{"type": "image", "image": image} for image in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    formatted = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[formatted],
        images=images,
        return_tensors="pt",
    )
    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    raw = processor.batch_decode(
        output[:, prompt_len:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    del inputs, output
    return raw

def parse_json(text):
    text = text.strip()

    # Remove a surrounding Markdown code fence if present.
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[-1].strip().startswith("```"):
            text = "\n".join(lines[1:-1]).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to extract a JSON object or array.
    for opening, closing in [("{", "}"), ("[", "]")]:
        start, end = text.find(opening), text.rfind(closing)
        if start >= 0 and end > start:
            try:
                return json.loads(text[start:end + 1])
            except json.JSONDecodeError:
                continue

    return None

def clean_page(items):
    if not isinstance(items, list):
        return []

    output = []
    for item in items:
        if not isinstance(item, dict):
            continue

        text = item.get("text")
        speaker = item.get("speaker", "UNKNOWN")

        if not isinstance(text, str) or not text.strip():
            continue
        if not isinstance(speaker, str) or not speaker.strip():
            speaker = "UNKNOWN"

        output.append({
            "speaker": speaker.strip(),
            "text": text.strip(),
        })
    return output

sequence_prompt = (
    "Transcribe the dialogue from these three consecutive English manga "
    "pages in reading order. Preserve page boundaries and punctuation. "
    "Use consistent speaker IDs across pages when possible. Exclude sound "
    "effects, credits, signs, and other non-dialogue text. Return only valid "
    "JSON with keys 'sequence_id' and 'pages'. The pages value must contain "
    "exactly three arrays, one per input page. Each dialogue item must have "
    "'speaker' and 'text' fields. Do not repeat dialogue from one page on "
    "another page. Use the supplied pages in their given order."
)

page_prompt = (
    "Transcribe only the dialogue visible on this single English manga page "
    "in reading order. Preserve punctuation. Exclude sound effects, credits, "
    "signs, and other non-dialogue text. Return only a valid JSON array of "
    "objects with 'speaker' and 'text' fields."
)

final_rows = []
raw_rows = []
fallback_count = 0
error_count = 0

for seq in tqdm(test_sequences, desc="Corrected test inference"):
    sid = seq["sequence_id"]
    pages = load_pages(seq["images"])
    raw_sequence = ""
    raw_page_outputs = []
    fallback_used = False
    error = None

    try:
        raw_sequence = generate_with_images(
            sequence_prompt, pages, max_new_tokens=900
        )
        parsed = parse_json(raw_sequence)
        predicted_pages = (
            parsed.get("pages") if isinstance(parsed, dict) else None
        )

        valid = (
            isinstance(predicted_pages, list)
            and len(predicted_pages) == 3
            and all(isinstance(p, list) for p in predicted_pages)
        )

        if valid:
            predicted_pages = [clean_page(p) for p in predicted_pages]
        else:
            fallback_used = True
            predicted_pages = []

            for image in pages:
                raw_page = generate_with_images(
                    page_prompt, [image], max_new_tokens=350
                )
                raw_page_outputs.append(raw_page)
                parsed_page = parse_json(raw_page)

                if isinstance(parsed_page, dict):
                    parsed_page = parsed_page.get(
                        "dialogue", parsed_page.get("lines", [])
                    )

                predicted_pages.append(clean_page(parsed_page))

        assert len(predicted_pages) == 3

    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        predicted_pages = [[], [], []]
        print(f"\nWARNING {sid}: {error}")

    # Always use the actual test ID, not the ID generated by the model.
    final_rows.append({
        "sequence_id": sid,
        "pages": predicted_pages,
    })

    raw_rows.append({
        "sequence_id": sid,
        "raw_sequence_output": raw_sequence,
        "raw_page_outputs": raw_page_outputs,
        "fallback_used": fallback_used,
        "error": error,
        "predictions": predicted_pages,
    })

    # Save progress after each sequence.
    with open(SUBMISSION, "w", encoding="utf-8") as f:
        for row in final_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(RAW_OUTPUTS, "w", encoding="utf-8") as f:
        for row in raw_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    fallback_count += int(fallback_used)
    error_count += int(error is not None)

# Final structural checks
assert len(final_rows) == 15
assert len({r["sequence_id"] for r in final_rows}) == 15

for row in final_rows:
    assert len(row["pages"]) == 3
    for page in row["pages"]:
        for item in page:
            assert set(item.keys()) == {"speaker", "text"}

print("\n--- FINAL INFERENCE SUMMARY ---")
print("Test sequences:", len(final_rows))
print("Pages represented:", sum(len(r["pages"]) for r in final_rows))
print("Page-by-page fallbacks:", fallback_count)
print("Sequences with generation errors:", error_count)
print("Sequences with at least one dialogue line:",
      sum(any(page for page in r["pages"]) for r in final_rows))
print("Submission file:", SUBMISSION)
print("Raw output file:", RAW_OUTPUTS)

print("\nFirst prediction:")
print(json.dumps(final_rows[0], indent=2, ensure_ascii=False))

Corrected test inference:   0%|          | 0/15 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(



--- FINAL INFERENCE SUMMARY ---
Test sequences: 15
Pages represented: 45
Page-by-page fallbacks: 9
Sequences with generation errors: 0
Sequences with at least one dialogue line: 6
Submission file: /content/manga_final_submission_v2/test_predictions.jsonl
Raw output file: /content/manga_final_submission_v2/raw_model_outputs.jsonl

First prediction:
{
  "sequence_id": "seq_14469aeb38963085",
  "pages": [
    [],
    [],
    []
  ]
}


In [ ]:
import json

submission = "/content/manga_final_submission_v2/test_predictions.jsonl"
raw_file = "/content/manga_final_submission_v2/raw_model_outputs.jsonl"

with open(submission, encoding="utf-8") as f:
    predictions = [json.loads(line) for line in f if line.strip()]

with open(raw_file, encoding="utf-8") as f:
    raw_outputs = {
        r["sequence_id"]: r
        for r in (json.loads(line) for line in f if line.strip())
    }

print("SEQUENCE SUMMARY")
print("-" * 75)

for row in predictions:
    sid = row["sequence_id"]
    counts = [len(page) for page in row["pages"]]
    total = sum(counts)
    raw = raw_outputs.get(sid, {})

    print(
        f"{sid} | page lines={counts} | total={total} "
        f"| fallback={raw.get('fallback_used')} "
        f"| error={raw.get('error')}"
    )

    if total == 0:
        print("  Raw sequence response:")
        print((raw.get("raw_sequence_output") or "[EMPTY RESPONSE]")[:700])

        page_responses = raw.get("raw_page_outputs", [])
        if page_responses:
            print("  Raw page-by-page responses:")
            for i, response in enumerate(page_responses, 1):
                print(f"    Page {i}: {response[:250]}")

print("\nTotal sequences:", len(predictions))
print("Sequences with dialogue:", sum(
    any(page for page in row["pages"]) for row in predictions
))
print("Total dialogue lines:", sum(
    len(page) for row in predictions for page in row["pages"]
))

SEQUENCE SUMMARY
---------------------------------------------------------------------------
seq_14469aeb38963085 | page lines=[0, 0, 0] | total=0 | fallback=True | error=None
  Raw sequence response:
{"sequence_id": "seq_01284567", "pages": [[{"speaker": "character1", "text": "IT LOOKS LIKE THE POLICE IS GOING TO SUCCEED, HER OUT-OF-TOWN FRIEND IN THIS PLACE."}, {"speaker": "character1", "text": "THIS IS BAD."}, {"speaker": "character1", "text": "THEN JUST... WE WILL PROBABLY LEAVE."}, {"speaker": "character2", "text": "I GET THE FEELING... ABOUT SAICHI."}, {"speaker": "character2", "text": "THAT YOU WILL BE SOFT ON HER..."}, {"speaker": "character2", "text": "YOU GOT SOME REASON FOR THAT?"}, {"speaker": "character3", "text": "THEY WILL BE SOFT ON HER..."}, {"speaker": "character3", "text": "SOFT SOFT..."}, {"speaker": "character3", "text": "BUT..."}, {"speaker": "character3", "text": "I
  Raw page-by-page responses:
    Page 1: {"speaker": "character1", "text": "IT LOOKS LIKE THE BAN

In [ ]:

from google.colab import files

files.download(
    "/content/manga_final_submission_v2/test_predictions.jsonl"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
import glob
import torch

print("========== MANGA PROJECT RECOVERY CHECK ==========")

# 1. Check whether the main project files still exist
paths = {
    "Dataset folder": "/content/manga_dataset",
    "Training labels": "/content/labels (1).jsonl",
    "Sequence metadata": "/content/sequences.json",
    "Training split": "/content/manga_dataset/train_val_split_seed42.json",
    "Previously trained adapter": "/content/manga_qlora_adapter/adapter_model.safetensors",
    "Training log": "/content/manga_qlora_adapter/training_log.jsonl",
    "Test predictions": "/content/manga_final_submission_v2/test_predictions.jsonl",
    "Scoring script": "/content/score.py",
}

for name, path in paths.items():
    print(("FOUND  " if os.path.exists(path) else "MISSING"), name)

# 2. Check GPU availability
print("\n========== GPU ==========")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 3. Check whether any project backups are still in /content
print("\n========== BACKUP FILES ==========")
for path in glob.glob("/content/*manga*backup*.zip"):
    print("Found:", path)

print("\nRecovery check complete.")


========== MANGA PROJECT RECOVERY CHECK ==========
FOUND   Dataset folder
FOUND   Training labels
FOUND   Sequence metadata
FOUND   Training split
FOUND   Previously trained adapter
FOUND   Training log
FOUND   Test predictions
FOUND   Scoring script

========== GPU ==========
CUDA available: True
GPU: Tesla T4

========== BACKUP FILES ==========

Recovery check complete.


In [ ]:

import torch
from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_PATH = "/content/manga_qlora_adapter"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(ADAPTER_PATH)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

model.eval()
model.config.use_cache = True

print("Model and adapter loaded successfully.")
print("Adapter:", ADAPTER_PATH)
print("GPU:", torch.cuda.get_device_name(0))


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model and adapter loaded successfully.
Adapter: /content/manga_qlora_adapter
GPU: Tesla T4


In [ ]:

import os
import zipfile
from google.colab import files

OUTPUT_ZIP = "/content/Manga_Transcription_Submission.zip"

items = {
    "adapter": "/content/manga_qlora_adapter",
    "training_split.json": "/content/manga_dataset/train_val_split_seed42.json",
    "test_predictions.jsonl": "/content/manga_final_submission_v2/test_predictions.jsonl",
    "raw_model_outputs.jsonl": "/content/manga_final_submission_v2/raw_model_outputs.jsonl",
    "score.py": "/content/score.py",
}

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for archive_name, path in items.items():
        if not os.path.exists(path):
            print("MISSING:", path)
            continue

        if os.path.isfile(path):
            z.write(path, archive_name)
        else:
            for root, _, filenames in os.walk(path):
                for filename in filenames:
                    full_path = os.path.join(root, filename)
                    relative = os.path.relpath(full_path, path)
                    z.write(full_path, os.path.join(archive_name, relative))

print("Submission ZIP created:", OUTPUT_ZIP)
print("ZIP size: {:.1f} MB".format(os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)))

files.download(OUTPUT_ZIP)


Submission ZIP created: /content/Manga_Transcription_Submission.zip
ZIP size: 66.7 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>